In [ ]:
#@title 🚀 **Start Upscaler Studio Mobile Backend** { display-mode: "form" }
#@markdown ### 🌐 Connection Settings (Optional)
#@markdown * **Default (Zero Setup)**: Leave blank to use Cloudflare tunnel automatically.
#@markdown * **Ultra-Stable & Unlimited Size**: Paste your free [Ngrok Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (from ngrok.com).
ngrok_token = ""  #@param {type:"string"}
import os, sys, time, threading, subprocess, json, base64, zlib, shutil, socket, tempfile, urllib.request
from pathlib import Path
import importlib._bootstrap_external as ibe

t_boot_start = time.time()
print("=" * 64)
print("⚡ UPSCALER STUDIO — ULTRA-FAST HIGH PERFORMANCE BACKEND")
print("=" * 64)

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"🔥 Hardware Acceleration Active: {gpu_name} ({vram_gb:.1f} GB VRAM) — FAST GPU MODE")
    else:
        print("\n" + "🚨" * 32)
        print("⚠️  WARNING: Colab is running on SLOW CPU (0.2–2 FPS)!")
        print("👉  SWITCH TO T4 GPU FOR 30X SPEED (15–30 FPS):")
        print("    1. In the Colab top menu, click: Runtime ➔ Change runtime type")
        print("    2. Under 'Hardware accelerator', select: T4 GPU")
        print("    3. Click 'Save', then run this cell again!")
        print("🚨" * 32 + "\n")
except Exception:
    pass

# ─── 0. PARALLEL BACKGROUND CLOUDFLARED DOWNLOAD ───
def _predownload_cloudflared():
    bin_path = os.path.join(tempfile.gettempdir(), "cloudflared.exe" if sys.platform == "win32" else "cloudflared")
    if not os.path.exists(bin_path):
        cf_url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            if "linux" in sys.platform
            else (
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
                if "win" in sys.platform
                else "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64"
            )
        )
        try:
            urllib.request.urlretrieve(cf_url, bin_path)
            if "win" not in sys.platform:
                os.chmod(bin_path, 0o755)
        except Exception:
            pass

threading.Thread(target=_predownload_cloudflared, daemon=True).start()

# ─── 1. FAST DEPENDENCY CHECK & UV ACCELERATION ───
print("\n[1/3] ⚡ Verifying backend dependencies (<5s)...")
deps = [
    ("fastapi", "fastapi>=0.110.0"),
    ("uvicorn", "uvicorn[standard]>=0.28.0"),
    ("multipart", "python-multipart>=0.0.9"),
    ("spandrel", "spandrel>=0.4.0"),
    ("static_ffmpeg", "static-ffmpeg"),
    ("qrcode", "qrcode[pil]"),
]

missing = []
for mod, pkg in deps:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"📦 Installing {len(missing)} backend requirements: {', '.join(missing)}...")
    try:
        if shutil.which("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv"], timeout=30)
        if shutil.which("uv") is not None:
            subprocess.run(["uv", "pip", "install", "--system", "-q"] + missing, timeout=60)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, timeout=60)
    except Exception as e:
        print(f"Notice: {e}, falling back to standard pip...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=False)

try:
    import static_ffmpeg
    static_ffmpeg.add_paths()
except Exception:
    pass

# ─── 2. SOURCELESS BYTECODE PACK (.PYC ONLY - ZERO .PY FILES ON DISK) ───
print("[2/3] 🔒 Initializing compiled backend bytecode engine (sourceless .pyc)...")
PYC_PACK = json.loads("{\"server.py\": \"eNrVPWtz48iN3/Ur+jhVGTKR6Uec1K6utHUeWzPrjcdy/NhNasrFpaSWzTVFcknKj/j8O+4H3R87AP1gN9mSHztJ5ebDjNjsRgNoAA2g0ZxkUeRlzeLqIZsmeS8Rj9eLeKp+/1LlmfqdV+rXr0u+5Oqh4tOS1/pddb2sk1Q91XxRzJNUd66vSx7PkuxKNSyXyaw3L/MFm+ZZze/rNJmwxMBLNi/iLL7ipehaxPW10e8EHsWL+qEA2Kp9L3vos4NkWvfZUVLB3+OiTvIsTnsanbycyqHzuKrjIlFj/R6DPwe84Nms6tPDR+ixd3IoH4Ao+SsvF+LX90AaL+Xv8/OT0f2U04yi6ZQD3wANergo0jyeNVB+4pOzfHrD69bjQVIBBzIORPQCC9NwkcxmKb+LSx5O87JSmO+PT88+61f2kJJXRZ5VXHdGBE5lY5/9cDY+Vk+93jv2Kc+vUs7Okqts4zBj/uEBO89veBawW14m82QaI3ViiivqG+bxsr7eUeCTWVTjAFhK2SFSTdYoHBTWZZxVYpgYXQqGVcZo1dQTwxf5jKdVOIkrrgadj04/Hx7vHUVn53vno7M+O4+rmx/ySZ/9kk8iS4zk6Bks0rUafjA6Of8++jw+GB3B2B/3jg4PItG2Pz4an37eO4HmosynvKoiGhkB3Ba8LE8qXiqQCsjx+PBspEE3MKh3B8qCz5I4Ql3Si3X4ee/TKBr97RwxOzwYjem3NQpWRXPi9PCjnC46HX06PDs//buiSL86s1sujs4PT44OR6cGggiyg92yqKZx2tB4cXK2v3c0Om1x7nzv9NPoPPrrBTydH44MqBIAAe719o6Oxj+NDpCe0fHZ4fj4jA3BsNR+Q3LA/ptaGsKD3ue9vwHs0QWM/GH8AcckWe3nVciz26TMs/AKBngNbnZ3r8+8b7xAgNkfH+9fnJ6Ojs+ji5Oj8d7Bi6B1RyHQHQJKvNw7PY/GP45Ovx/tHUQf/g4CCWC3t9jv4a+dXfUPe4dtnz+wHPTqGqwIm+clWyzTOili4O4kX2azuExAb3/H5glPZ1Uv+rC3/5fR8UF0eAySfrw/ikA7h2RPQ/xr1w/Ca37/ZbC9c9nrRSen4/3R2RnYBuj0SDbGk2vgDVyrIuyQh6tvdFDCIN9K4TU6GOKs+4CSWD2k0vR7T2hkzqARrAhDG8CEtfAzDoyAEbAAfIbGZgKWBsx7zpa3CRi7jKX5VRX0hCmJUjCVQJbeXMIjaPDh9RmQDNIUnY+BVQNW1SV0W7miexeg5tTVCxgsgNzZQjHJskyreM79nd0AJHbG5wxGR4i1wMIP2MZ3OMWA6L5LgBwDP9GKfwDmEiiwkZMgKxsk/U14E/DjPOMCzlWaT+K0DQPfJHOW5bVkJBChH0KAkhR+YCASw0qxH+N0yUdlmZe+t9eswWJZ1TR4whls4vVD6AXPEGZjgwtiztvDxR7TDDvhFu5wbD9NeAZ27YCRbZHbzX6aL2dsHzahHJ78ZQUiADtAfc2h54bcT5YzGDrlQUgSVLPbJKYen8bjT0ejaB+MGCglQJZLvcCJbkGH4glsuYyHVyGoN8ySxpMBwGCGWHzx2lC8S6DG+/v44jT6afShaQ/joqhCsTcBniU5K1kN+/HC67WBDLT/8QVYcumQxM60JIa46lI8aNt9iFpbqa9+RLe4lo286PnQCcJJ++gTXV6KJfM870eCx+L2Tg98k3zOZkpekbtIIyoyunR9hRuKXJLB1GB5egT5HLqK8Qls3qwCqLCGP/x0Ds/VEn5OHuSMIciBpIolsNNj1ximwyW6L5LyATEgmGrFmU+SuYjr6XVntYOQjTNWLadoadgdb+P+vpLzEshqOdnkizhJN7N4wTcLYBLMzKqc+lfSLk3jDHUgrqp8msRoj0gD6uu4VmyLwUtdwrIrtpqKaK+N0siObLStAy06PtflQ/Pyuq4L5QOB/LScolB6mH6gByTZPG86KlxCKUfCWZNCpMfQOAvrvjVxv4O9HmpMjAtd0h46z4V0Q5NndJirPsQlMPqeZKNSKNIi2E5x7mqwuel6bVgzF/OMtkerowdCfJuAtw4bkyfgeX27h+YZdGmIAJnxglZPkiGrl2hp90Mps7pRQ7uXFEOro2pzzhwJZ5wjopM8T/0WHs37wBj/RL84hShMRyoNO9+xQ6HTpk7egam60qoodZTP+qCOKbgsC/zJ62nolGZhwkCKwK7fcrHNoVL6oGGzZAaq1bKQL7NhpwIe2JkJh4CnZBoc+guxcCzAKGO0AsqLMyozRR4V6nrKr+Lpg1Z5YbvmSYnCjoMbC6ksYwjhHJKG9g1BEsAZoMh8Mg8x+/lnJWM//yycNrDm2jg11vOO4Dc4IgJzWLbKaU4aXq20GNAzkgpu6HDD5aAz9NFSB8kFT0iI7LJi52mAquVdN7NrfVFiBy+j8GOcVlyKbQERMazmsOOEmUhjJgPNBDjRPJolV2gbNXjll/Q1NE0Dv4dYdFpHQqB8BJ+XyT9obdbKKLZYxNhDVy4ZevkVEGN1VwiGVZEmoMwMjOF2oKCnQCwNC9hwyHZop6bnL1uXYZrfAd70xhNEeJ3JReftS+2fdfWVEjBMubrTZVmCjAqdlXvBQGU1XqqrejvuaiWL6zqeXsNTLXZgtdFVNSqzr3wNKZ8beZY+BKHSD4k5IApgNH4hjYVNBCcA/iGMwKIMO4INIxnyJSSbtL70P1YLAaycyP34ADGGsG1I84iB9yqqKiPpzr9sbMvjb6zacL2Aks/YnpRAEJOHay2wFi/qC95bg0ATMVi5LR85vKyiaT7jw92t7T5wtQbzNfQuMoUXnw2Uh4jILWDjxzSdZW5VeGEtXChRJgsrlXMRI3GYPYsmDzWvRNwFkaJAczF5SeAuovXo8wd0MP60tYURuyFFMIcPlCwmgR2pAw7TFDxBmb7bmxEpeQbBSl3mKbBbi/o5BaMbGDKyWPWjJCd0FBE98gAapF4B0hvzNLm6rpkgryKdxpRAnKYi58qwnURe7DfIjyjJkjqK/Iqnc9yI76MGJrK9Nuw99gntLsAtRa3VHtiDQNoSEBmF2ZBt2e9XxuAa0XhKikZ4tiy/DisbULZfJ2f/JZ/g1EYiT+GVlxExaBaRj2h4wFKeXUR8N3QxxJ7Zuf08A/cPFr4wSyvx9IoZnOCHbNvl9Z6XS96wu+QpjyuD3bYmr2X3Kn6xrS7qro4biKHGRKyIxiMxWbwWDUmXYwbQw0grVTTV2gfCsVIzfXfCLlil000OfWA5YqO4TB/Y3tmnQ9Zk4BkHn7ucgtyLztshO1xQChcMNlhFoOUX8DHItwMLuMys7W9zApZCOJ065w174GZcJJu325soRpsQeCySWvHnw+jj+HTEpK5RsqoAdcLfuHOaqcPZg3Akd0KJ+u72Hw10yP3cF3mLjSOeXcGCYFTAYR1BMRjYHnAmZgxDfggF/sFBvBvwKmEppvijnmLn2/YUhsVRYgJYC6s2jYt4mtQQ7GP4jbEGckXA3AVPe5llSBma+w2SJljpNAF2kENdAxeA1ZvT62V2g76EBA8cLEp+SzNmZHA5uUi4pShuiTn+FLJ90JbMMNRVmtdKh3AWNMHSdd/kmCzbnOmTGTqPMtz1VbY5LgpogI0NE1ctowzv0P8zeggwjaMSRbCfpxoYTF6AY1PyKQfFgGeezQygqMLY44tXPxTcu2T/MRSRtNeyrHdxUmsUfCdUh1L2dBvSjsl6HCiDVWjBbdUI9hccHIGZ3U20YcdPo3MvCJdFgT5rzyRBAAfMHarg0faoIEOfk/HZeYu6dxDZkqfErsnHqkSikZSXULF6qz5DCuR8A1f5BpD9chm0+BHfkSMFo2QvGjLxLKcMRk6AIeGMo5/keym0Zhvb0EzCVA09CLJzjPA7wO/lEVob/v2GcvGE6XjjFOv8SkUbuZS+gYyKGXCNTBw57F/Kx7bXYVuZBuIVePlg7KWJgLgdbCLXxizOHshskb5X7Y0Jo6qXhreNzOKimqebfqcP/nn0hO+KUfBrvdenvhNk2zvudAo6LUIlAecXaKNLI0nhwKURJ4jERFzbrt/cGVHndZzqEW0Yf2CrzrTai623mta+QsveEfBp2pFtmUffSGmc1/HmxLDuQlvJUkvKUxQTGRmI0VpR4mqaJF4QOAfCZHLsd20WuWd6jbw55G7uCTfE3IOTxXKh92HagvM5e2wv2OYm841QJXhinz+sFEuneG7/cXXnYOWb14prS2zbzTIj2ZxKudlcgMfWFjtwP/Y6kZY2NbDt33LpHND27rQsLq8yVKHLVzItZ4hJySbL6mGgEGo8DzAxn04u2p4RGEdMjYTg5YMNvYoTcFHAOtXpw8tMz863/yrT8w5dtjPplLFNptwylFLhtQHteaZAt8xQKXw9bYS2bOCNOyQdukiD6a5NlmcQVIDqWkA73RbVFbpeknQXUlJAoKN2p4bSnQqlw+6tsDtIfES72RDHC3cCn+V27RxlMwHiPUz2NaBWWip73GsMliOv83XshnCFKSHUMm0gEr/Rtr3GWqnETnVlC5Rzx3A7xS2Jc+nEPMmAFgdEt11RMfp6D/tFLjoern+QMQ6aD37Pp0uKvoQh+R1DpwVCOOggc653eXkDqigyNr3oqliKHApYKPo3/Cv+/UXWT116IMFGuw9rhOs1bCU3gl4kAK+uypDvQYZKkcMXeQ+Z30M8NIS88FvJi7trDEQx19FwGaIC9HA0BaRlDVM7awyqAkPCBAQYIy1YC/LssE0INp5HtgrJumuK3RdxedMAcZgNXO0kW9qBhiyFySkb21TmENoItQaWR2hoOo5PM7KTm+2ghYdIgBOoXXaT5XfgqSqwA/ZoTfPkvRZxRJMq6kIIqONF1Yxvnyhi6QZ/lv1v4C7GGs9TD4GKzw3fzqmhhuQQT2bAWF+dBvGsWpa8JbT+inocu5dRL2PohBWkk9Nhj7Jxqy0VElllHzqCrAzbutJneKI89HRgCO83xHvYbmYxX+TZEFXH5iQl28t20rSrpiLB2PsvV11uszGnyRw8oTjzI7BWA1U1K7fnFcykdw94RopnM5QKkeP8OqlTIOlCFRye1ctZkrMP8RTCrhmDLkAauFNoW4feTogBr0JhqH4EaB4hGJknV1jhgSWy4NbzOIMYBdcnX9bwbzqbxuUM07e4H0XTEjYpUAIwTsS01SDwnBmHsEU+QeuEFGBy4o5PoA/WOFW9CGt0I57dAm1fch1AYzYwR3Oz8rgCZ4rGp4efDo+puvD3nj4R7HsUgWtol8i7MJ7NoiYzKfZxuyhYnmgRlRDhXiVZNdT4mS9NFpCRNl+K1Es1/AIoXZovZCynXgQ9F1ors61mdR3Kdw2bJExVPgids48W9WErVXKH0+UsRrsS34L2Y6WX6RaC8SoqqkrTXXGOGb9NpjzCt7ysE/B9twxzUXIOhC76TLgj8NOGAA0RQsGaC3Ogu+pE4wVhAIpUq5xDoCJqJYsqRGVu9bgFYys9o8UEOpa45fsNbi0/CQ+MXRAEWQ0AReb68VZBQEOZRZUhJTZJ3j54JfA2nYDmGq5bh6StcKv9tkFXv3xSFXGYlgF3JiLDLTYj39joSFIw4BevBi35WSlSnudBDIP1HCTTaVLV4EOVcXbF1Zy4uRXoQKnZVCILE1iULpexXV6qjRNNuFHQ0YzEcEKX4zZpUyy2xqxpLXJxggaZPsV3VG18v1EtZ/nGRVqX8T4WO0xrM5uBgQvBkXVWrVLt1pHLcwe8Ozt9w58Ha15g/Tdl3aWBFpO9f6R/n96Hhm8hti3YaYGB9cMKwuxOSOHeEgysfxzjCVAApFkVCwT31yjLS9JMa7CqgwBHu0jjKfe9//0fBHhv4CSmryqKNtu4iBdN3YXTm9SjVW5JNHRcIv8cVppSGn0jvRGsALZtLqAiMMFand17pGHXeyLBEv2/G7Kdlt/Q5rW3C8Q3OKVtsDv3QprAh0BG40NMjNdNgff03Bw76+dAgA1gBbWPpXE057PwbVGwWGT3lcLuvobwtWS+Nef7R7vBFn7KOpZJVTjkjNoj8ELSJEPOkxVt1UqqcyFfA+pToa9NjAVJzkRtuppI1QbRgqC35NWwE+G/DyTqni79cnvZ7Rmo5FBP090BpaUaCMOA09kKPmixEXrY7BjYUz713OZT3FSwbWd0wx/W2U98jyTTtYbb3fDPLsNJQKQ8Oe7UfC1hQtC28cSJWwa0bXvESSy4mKVlfwwydQegcyd4m1F6DVGeoKPBi8q1sYA6QwQ5BAtgUvpslyogvwlCuybYGGipcPt+0tflejPpY/P76T9VYmrABMLfmMjiEq2RLRQrpLcs8hT8E4dguQjEK1lfXaCkPD29NwgyUOtjNhzc+ZyKDr1VGky6MmhUAugzhGtgMBHfIAdE51X6qu8Ovc7d2b6PYvC+wf6XUQyhCZqEap2/47p+97VYrK/6rXF63HZwNVfottTreCKuVIHbWV9j+vd2J6oWsNLruGLec/x63MCLlGtYMc1TcAXiYgU96rVQnTkvsxy9PblhmaRoQK01tu5pfl2y9JTvH9XPl6wz7maKrIEGIlf/+XLKLQunThoPPQ6Vw3vvBZScgVCbuLkJz0aZa57LIFXado0unovJEG1FkgXXQ2ZYsJfIsVB9QySrZ2cynn2yMZBFG9c8TqkkZEbXqXk2hSh7+EVervbNCtzg0kRaDIzoCK+NdxMiCo4hZrkdWLZQbt5MBF261EZc8nDdqDQGXRVL6OTIShh9jLo/6PuSOsXuYFm+hAg5Dw26AwlipI4KYWArJd9EzLQ2RV4Zi0MVNSLHuiFyVRtUXyPuALx+zQSoSICKGlB+pwW4PjA8WFX+7+imbk253q261vgyrfKc05FR4rWogFYirnQ9v+nL469WKeqrKFdoB7Yyurqi3VgDoW9ogPjh1kKUjwlsl5jiesO60g0AA8Q6fUTjgei0sl/NfePHTt7eozwv6uru39jZsuDlBl3sWYrqqW5/nmHGaeZMpFEHcVPdORn+AQ8GXqk7WbDa8RfxcIkMlXiKZvF0+eSEgwlc8oawryO/EiY1X1SOkyHHqb0nQiBcoAH74m1vfbNVMP/j9wcUI+/8hfnbu7tbBT3twtPO9p/lE8bdlzbAFnx1k3sd68lj/PPW5vbOFvt4cvYb2X4zYLeKp4JNfXa7IohSXHIxpXEsiS3SFb9cMbdgnel543o2Du5zbGqutK/j1N4h2wPnc+MIY+AD6QX+Rn6tcG0H6PNu7MkXGzgtzloxX/zzO3aMs7NTvshv4zRol4N0CRT38deRd0Auz2fwdoQn8M9QwJVeq55/T75iP+6wM3wFuhBXdbDi4N8BED8M4ob3Aet6/e+Tq2t2UvJpgkbVBdglkMqLI4tLPqLvdECDlQvx9OyOLGtcX2+nxUB0QcTJDyaiB8bnZvBoDf7xwzCU+NnZcnwP1BnvhYse4aeAuneXqK9x28m9y68b1Wz/TSQk3XrjbPxt2/vLnOYmDew6IsDvZTz1dEGy5gX6BognXUuTt+/UO328h1XXRjTnyB2XFZ2t0ihy/HwDTiddQ92w2uyAihcpWeM4YX8dj9RFX2uhH3m4qK6e7OwMrUuTiRTY96lcOng7Bm4E9B3PmMrrWD7BYuGwk7rHIcBAgYtYTHUoNGterz8nMo+IpEC8Y/sYdrTr8OQRz2Q5h/gUbQkd9NQ58KC60ddvm3qC+TJNX+Wc7nzb8MUqFMSqHix0EQiB8CFoZz2gpAAP2ckCAvn4iazVN+J+Gp/+JTo4PAVzoz7YhT3w9yyB0Ft6qbJ6GJoAYgN9U3t2pQ5eWv3DxQ0Cwnu4WS1O0/HabVLVUX4jKxK0GvL7WmFMmOBf6EVgFOAJmOGi2MVjcFiF5N5KEOB9OwlBwZL1wvQzY45PDVHVukcwhUEEuiOaccjm3mP7kzpPj2qSJ3mSlxXLOpIXIQwmbTaQBHVY1vZcPbY4Cb0rE2B/1hRAUmkUtkKgh4WA/hwLOurYEK15SO99au51Px5BhSh5gd8L0QiDb3Q38QI0ITk0zW0ldtVaWZWNum6SFopKU8zjY1cRJcVxONZdjTgBIDfdum6LKWZBpHMOu/t3Dd/dcxLl4TTNK1fhpzqA0TwLSXArV8Vp89kN3XmZgRt3swLuq+ouX1xz6a63fKaU/PWFlqvKh+XnBMM6j0Thkm8Kbl+wu8+M5ZObm8WHlsnsvaC27CWrtG5lnrPNf7J2zY9UYIamX5ZiStXHrVNtm51qM0NblLz1nEI7HJqXLf/ZlFm7sZAaLM/Fyx9ig4PtRn8ESvp5PM7QzcN8yZA1UrsuVSFuUzgSLug+rR2oLxb1GskjYAYWCMTMxySVCawpIzQHBVYRaZPKybNqueDNIDudY4x/1RI5i42fsxVS6MdnK25D0E2I9WVHSBrlkZCDrXCwyZ5qYttfwlnwqgK20NUFzQbWlM5OHuTiKe/IrHYxg55eU6xrcRuMRM2bkMUKSprxeGTQMMsIrtruXn+tMA7N9etLefonCZTMCfr/rmLSSAYiKx76bpmQq7eugNPC2/CBEXEIwsDyYdmwZfJFh4/gydp8M0t4bTdYuL/NJWEv+NdZyVf65jq2fCG3m/pzdwaXEgOPYtzTphz2tkwuAhEAfAHPzMR3NVQOkX0DM18PTa+JzXcbDv4A0yCEOZYgelYivOFERFdxgzWZEs2QIl5W/A38oHEGef8mrMBzBvXJHfO8ISR8fWvm/OY1E39reDH7cYYACCZRCyHS+0d73qf34hM8odc6rdCS2+r/ktUqOe6wb1guMfD/0XoJhL/yggmg/9IVE3vB248E//+smLVrf60VE0DfvmJqLwavyiNXxcZcpkxbt5aE47xisddsLrP8LsMo4A2rrYbiZ6aWaf3vteTy6pHkGWZn8fNwKQdPwlOnzNgBYlTlJ7x12ccEgglljWcPDKmTQTwmoY3d/n2gfQUKjkWanjJgLWRskZR9tauDFOhGhI0fj4NlxzjfDihfE+p6J8Kdx0tygiYVEqpPIcC0FB3S51jIL9L0iA8IiPIm5iULWO7NQkpwg6qVx0Mxbr77LbNz+HXCfFOn6NQHk4zP1/vInyEGCQpu0GcqdzjUc9GlCgOtYfMzcCoGSvjmm3YqqpABy49fLxus/Dge8EWBan9lLwhcX86T39/FzA0+ygHy0xrOD13+BN56JfJ/kxz+cn3uUnxtbMV3gFvfonRcATGn7hwJmp+XJG6gxDw+BbLIXr3sN9+eNMtb6NN9A6KnVclyxycV/S8Jq31kvIZuFUKJJrzucwUGqvI1jIHxfzCwrtGixRQ3mzD7rUaF5ocrWt98UQld/EjJcyM7X3MxB0NYUT5YY6nFvjZhjHruW4ESG7IXFnr6meDr44x1HwzsXkQVOa4GV5HoEhZyd2u7XTHX+417Qnu6eIoGzQjr2h3wwjSdMfmP4jY/7K30iRzPCn1F9IltRv7D3lOegmeIdtO6FmO8x4lpdZlERS6oa/3iOjT1+lV6KtVyUk3LZMJ9fOnK+r8D65Ji5QBL9AfJqiwuqmsgo8KLi7Q7beh9cFMQLsIpSoBUsKdn72sy98JLMb4vneDtQOU3KT9Dwo/o61svWgoJqBXMU5uUb1ypQHwoWuGKi6NXrklVOD8K3dynX3WqIT4Vpk40fm1dIl+PPo3t3Nam1t+Kvn0qIvMmjv8fxrjoqFI9nfQzLtIya4Tm16D3f8vmA2s=\", \"tunnel.py\": \"eNq9Wt1u28gVvvdTTCfYDdlIFKXYjiNbKWTH2TU2sb2yjALNBsSIHEmsKZIekra1hoHe9LoFWqDAosC2F32HPs++QPcRes7M8Fd0srsFqgAWOXPmzPn9zplR/FUciZTMWMJ3t7d89eZH+VOU5E9xwNJ5JFb5u+D5UxK5Vzwt3rJZLCKXJ8XKZF08pnwVz/2gWJouBWeeHy6KAX9VTGYiCPyZFTORNMcEv854km7NRbQi6ToGDkRTnMWpH4Us2Nra8vic+ImDw44fOlnCDXweEj9MTdJ9RWZRFAy3CHxu/XSpFbHUl6Hfxm+ck9PjaSefvTg7+sq5mE6Ox+9MwhKSKAb4SayEp6hBlKWGbe2YxYzgaSZCIHCjMORu6vA7w6D9wQvLhn992iEomGmS0YjYWvK5H3rOXHAuFTCSlIEahfxkRPZs25ZqwKsSQhpgREpSpdsSLN5mCLMUXa58NiL9rYq4koMSxvEE80MDPSu3PI1CrlanYl2ygQAhDshDkNBKUg8sUc7KjRgEBj7wO5fHKTmWX+CxiixIorfF8HDA6Q4ylNuDqViachEmHaJtPSTzIGKo+S6YU8qXR8H7JBUfFGtpFaDBRRb+MZR/3GgVg4E8mHovuKVfjdiU2sRSG73jh4o9K2xIVzM/KCQqlAn8kAPnij0s1AmHjTI+/DkJo1QRs9BT5HEUBMDcT+RcafH8MwNGV1UWuLxOghq4qEGuZH0aPyuQDkTjTLhLAzmYGyTAerW5sBIoK2shoiw2bLMaPiix9qOKSDeIMm8eMME9J80gEQKjNZ5bnEcpfR3dhuBmj3yurX1UsCNTyY0YlR1Mi0ykHAmgDCdxNgt8l3w5nZ5fkMvJWws4qrBYJ4BK6CINcJYaMUwriG65AC/peEUZqcXvOEWL0FvIz+g2oWhezYQHCSea7wwSDeJmCYuixMIn6/cRpFAOgdYCsAKePR+26JA5rch+D5s9UL0x7gV+ye4qO5XecOeYHijZMk3jZNjrLQDKshnGca9kWXn0eoIHHPA+6YHGgKI9T1u2StSVW3bZytvdVgrxoF3r/4MselMljfSAlijh/4fdPSZAgNwUm6Cnkzf3Mr/zkzQxcv+b9cxJV7EDUyBmESDPIKpgmNbonpCTOWFQl90rzC3AhFw2BASFsAsBVbZDbpmfykyHrzTCuuEnS8LSaAUBv4pueF0AR9JjnakOK1D7mB4SmBqTWhs1pzkfkOf2JlpIuEwCzuN6ZWyIJUvQVgN8frp1ZfkQACfGnP74/V/+RXLQwAah4lM0PhNrNNd9zurBsiy6KVi947DgFeBN+PyGGyriOrlPW4GzTBjUoiVpqh9Q0l2uIi+3a4fY0YudFmsBoeAAWC4vSQuTlPZzV7Kq5TMdQhXsQr9Bu90w6rIsBdz2IPjVEKhDEYswgyCBigZleI9A/UA/lEUaKhS2GkW3Z51HMQ+Nmqiwf6c2oCrgqLrq5Px4g4YLUaW5mL4+u5zWqVKAyNFUZLw+PMvmif8tH/XL4dJ6Ch5ae4r3okCN96z77bj7O7v7svvh2TcWpHoZON8gmNAPRecxerFTK+LAtZHueYNrTeWTAZULgH+k+qkOgbdkpGSAIuAxvopCqZZpySJn1H2v3KdxDv4CkfDjBtET8lsmViSLqxWSewtOAEhSzAQIex4mGQz3WOz3bvq9JWcBQBFAS+DfcDLjgCdcF3Jsz5tNhWzyBAsX3NhrycEaPtYbhmsleTWnJurbaF0hd6T3peYPdZlp59FlQAChlIzu6WXCRXe84GFKh4RexonLAi7QTFnc61s2fWhlYraOyrPCJizI8If3Mjj6AHR4RACYjoePSglhgwTo7zRL8AQwsO3HyTc7v/zzeEf9KBBv1zXUjVtp61q6W1c+NqSt3TuqWanFOQb/8N0//vPvP1XjUPEmkOGRGJJ7XrQ5jzWNIZS/q8faRUigLF2m0RUPh/W2EQiQ1Uf6yQvZRC79xbILO838wE/X5BR3y4U0woj0bfvdIRwUvTVkxspP4fgBIQU1CGJJJZU+0sEOZtFUSolABKMQj0DSAGrz8MYXUYjNn0FPv5icfeWML6dfTs++Oj6lJhJRatayWtc/pWPzQKmstZFxG+mXn+HX0pqW/Isukw9bjQA6kcTH0kNbm5VVFdaTEIwGKQBoorlKXEBLQh8hrQqGwn6ahW7u9s0KW0F5kYXGe6iO2N+5meSDNWmFlSn2Y/zy1aayVl3LcbU1IrK75O6Vgs6foXpBqmbh/O6gyxxpbkP+NR83q1rk+YmOgQoQfzwZ1RG3juoQLopfzgzl7qhuljaLmLanOtXUMrWtCqkxVVEShC8jr/C0BbwbXTSFBhWG3r8Yfthq67N++PsfyZESGRorKC3VHIIUR8imrTjTLGA/D1RqqTpn8ghvAKDgYUpHJnbPKFGJPyoCPwo4M8Dzx/FGw9EvQpyxPPCliRZ97oskRd8wjRdQfcH3c38BldnbJxGcW+HkAUfKDI4omyC6iTYV4f4XvGlgjYqHR9C4gsCjRrq0RWLp+aoPPno9YGrvzDI/8KCP9SEMFo4HBcwBJ18ZZQ4MgZFQAn3SQ/Cu78yYYCsou+SeYvc7JCW/3C4PuWEqnCu3Vbj+PZXDFPcoqWqG1brSTHcfkFo603+jU6y47cR2godu5HFDsTfL+7AQWhrnWjjqxhbm13i+kao/Fnr106rCw2uB7EsMuhYguRq0vp4c4dY1x91g1YPeFA1Y77hkKXfcSAhVBUeaCygHfg3TxDqeTM4mztHZZHJ8NHXeNTr26M6RLftuc1yApqNBWyN/LSzmeQ6cW1hugNrkil1xY+6njXLgrxZSSTnv+Cu2QKogAOGDSIzok/6gv91/A4UFgaMYfSM/FQiDQwbw8SPrcJ3y5OTMqG1hJQzOhkDTIbL8pSN6fvoF3bgMVg60Zrvb2tWwBPP0hgUZN0zT8rgcpixxfZ+an7g13YQzqEtwRFw7ZYPiuEx4vzBhyvtHgJ2JDEPS7x4BpysyDj0R+R7BpMSLoyswUZoCgD+TF4/MTcnXE4La6It2qPcS32HjHPtCKCIBmxWgVks3hMb6BehGLZY/BUCbD4Gax/iiVs3r1woV3qMG4U+u4jUeZStWQBNe8nwStKrmVxtjcu9uq5NqI9sLBrqG6Y7smxARhI4o+TXZ2TWrUz98909yeX5xNH57PCEX08vXJ2fkcHwE8P+anFyQs9O3J6fHv6LVNXh18te/5Xeql5O3UMRLeR9qtEj6Z3JR8Sf0xb7LZPnmITZxUJX5asY9D57Ay/1uyuI8Pj6HuDCtOketBSoEepkt7a109cn5Ol1CZdNRnvv8y+m7t5089Kvw5syCyEWXUFqtUMrYdb9WiOeU1u/lDjz/BrJmHfARXcHp3Q+7aRQPSX8vvtuHYuDhYV8P7eKQwjE9Et+RJAogU570jweD5zv79NVG+1XdYR6FaRfREdYOkJvEpCF5MrbHL47sfaJFmEVgzhUQ2UDUwhM/Z9CduyxU1ssz9iLNPD8iLI6JcTB7dYEEX08OerNXLY3hQQ9EaxEYcTUR7ogiIA8lrvbicLGvgrZzr2z8QNvPtHCIHxVH8lygc5UxCBtYix5Zqq1063vpEpTfkQZfcjh8pMWrtr9gnp8luRm1n4ZEeq28ZAXLzuVnn5JeXdMW5TE2ioFlusJC4vFm1FT9Wa9w1W3tl/a4b++3lMC2sGkhKzQcYAw0Thta28E2cBrsNedX7K6rbbi9Z2/OYogpvpjhUX06j0htt8ZNEYTvnK38YD0kXQiygHfV/WeHHCKMvWPuhXx/A5Qd8vSCLyJOLk+edsgkgqCOOiRhYdIFXPfnTa3vusmSedHtkNjSreQ5OJyIxYwZdkf+s3bN/Y1bwy4L/AXo4wJScVHON9Km6jYNJ3gSwF/IuvOAQ9hIPl0fpE8KbmTBYh1VzdQctKfmQRKzsBnJdi2Q7ZY43rE/awRu3z58udffbxhGmyafpa8OerjhJ+So4s5z3F0O3GqBXtiAPNKUqQDvYJczJFkM530XMn6fBPjLlugCQ1eGnW3Ze3xVglchS2tWH4JSUPfIW7yI/Bxqi4vHoRYw2tRD52h9bDmo47WyCriI2FovpehgsKHoHiraCPAWqWWNrQNYQ6zloCFo3CaTzLC6UMr6uQgvD8eDw702EaZYU3kQ3RKoqusoEySG4sjxgCdBH38CzVswOBBjQyJzGdpTqNZBsLYaEsdNMzKyFHw+ovdFD/JAib7NplAvWXhFYXMXG8KR/tXDgioqWwFLrn1aWfsUELYNFGUDlaebrML7W5sXsDpLIAc2JmX4+9/KwNM5A0MtdJXcwaRmorvA5II8Nl7aHl90yJPdo52j4xfE/gye9w53jt7syk3NTW4fQ8ECeLChF9IeQxKCdzbJNtKsnULHxm7colcB9jgtEb/NRvWa+AhRFUckMzvH17691yEvBx0yeN7vELxHrq9vidAfv//uD+Ts/PiUnJyWHen4/LwReKwRePd5N/aw9Sg+b/6nhmpn1qpd1Y79NoLco3tjqMw7LUEINuzKi/ihuo+HuhJ80grVRvpj7UXjtdpP6OwwsNM1iq7DNH/K/6r5Lxqg/pw=\", \"keep_alive.py\": \"eNqlU0tOwzAQ3ecUo7BJpDYSEmwqZYEKbFhS1pFrTxOrrh3ZTj8gzsABWCBuwa3gCNhOmra0iAWziO2ZeTNv7Be+qJW2YPkCI97tK42EcVn2DqVpFUXFHLEuiOBLLHQjpcuAHG6JMBhFDGdQEGl5wbihSkqktlgpPUedcGlRL4koDLoIMyNwHge9uExHETgrhZoSAScahPDpxhPdYAjX2pVLZvHX2+v758cLjJUgU7hzmOGVxwCh1i9J7YFbMiN4+snr2aRxGkqGz6riAk81b0l7s3qzO3g7g4lqaAXjh+srELysrNgAnwFZEu5YCTzIdoFwtxltGMm4Kfq0JD2s623thm7TH1Erk5wPgOGSU8xjj++o7xtDAevei2uKtYWbsHAlDzvUxJjdXE4OmRFu8qPHS9u3NpZou3c3fz9yHMf3HgReJcOdSmBK6LzUqpEMGMGFkp0CHSAAfZVek9kk7BJXqUSb/yK5AbiwyY9IDdJB1yP38mlvzGZhmKQ9abSNdhS2Y6p6f8r/CLb7U74B53geWA==\", \"models/__init__.py\": \"eNpTVvDNT0nNKVYoSEzOTkxP5eICADbQBag=\", \"models/base.py\": \"eNrtHF1v3Ljx3b+C3T5UAjZynLsGBwN7QHrNFT3kklzi4h4MQ9BquTYbrSTrw44v8H/vDEnxQyIl7cZu0iL74EjicDjfHJLDsF1ZVA1J6rs8ZcURE68N29HuuW3Zpnu+bmmrGpqriiYbll8ebatiR8qkucrYmsjWt/AqGpq7EoC67y/yuyX5O0ubJXnFavj7pmxYkSfZkpy1ZUaPjo7OXr779Z+vX7yK35+9OHv5nqwI4PmD5jVtgk+LtNgBWEM3iyVZbBOWiac0yVOa4ct9CDg2dEtimnOC4zrZ0uD6tOMy+g2/Lgm9oXlzyok5r5tqibRdhOTJj+R1kdPTIwK/xWLxU4JoSZEDx5TU7bpOK7am1V9qgYFkRVFKaZAblvD3KIVecV0UeSxakIYIsHGsbEuuo22bZUEohsFfU93pF/xdR5e0ifPiNmFNEKom+jGlZWMz83JXNr3eZVLXRwO811HZKpycfIHYhfRnIFD35PiOjtIM/iVnSf3hl2KtZHTGeXyCTJJ/vP0XKasipXWNev93sSa3rLkiIGKQNPyTNBSISvKaoeZrkuQbklZFXT/hkhRC3dCM3dDqjsuMD8NVGrOcNXEcKLJqmm2X6g0Gi9nmFMfSHxsgNgYrpL3vLEdZoN32GsqkSnZ13zJ0e5oxIDEGU9gxYBJ0jGN2how9LsBo0YhEH0PJSG4kqAQQ8WA3KmqhXT3bIJpwgNEvNpBgAgDEg93o4gBAXZ/tjkXbaKG5OLbBq+QWNNbQakc3DPQ+uyNaSVtzvUD7gvvxZtHjsCouK7CyU7LNiqQBuKfR0x6fbVUhR1sQAagfSEGonsCLJskERO0B2ZZjg9AmiWuaFvlmDKqs6A2jt/H6+fdzBECrqqhiIKlOLumcDqwG4bY1BUtcF0UGMD8nWT0EUoFyDC4Fd4YQGyeN5ggnhQj/GLFIAHcRmYMrSnk/RavdJc6K9APi7OaQ6N0r+NLHLDiK04wmVSzigtnnJX6Z7hPhxBES8mcAQASv35wRIapeTx3bQZE4O53zKem8i4ov1iD9JG34uK8gVi3tgHmB3J5f6HClEAY8TPGZQU9DA3R87rEwGoG7N33BSNZ7sEs+1uwPujp59oOWiAi8SuL2DDFgOkrKkuabIMg4b9ehRlTRpq1ycq15a/M+d30Se1Pp/vSgNIMMKIFv8XVItkVF1Ct4qqMH23atNckLyFgMbcR0B5OeoHXO3D9NsEVqBgYTDCgyrBPCDJV2ZZuNsJoOjnMpFMB5NIzSGh1Y5ZkG+nRWgDWbyYQ1aKfX63DQDlGrYXlLrYZBJoI/b1YTWHkWUC2law8mE4x3bY4h5CVGt/nUola59Kweo7rxGtQACn+HWhmaGEIgcQPEhul181CdJ2V9VTQiMHFr4UZn26HmBJKfd9zzap58SjQyh0ogdSK/0/V7YJ42TzAMJw1bZ1SmUMhITm+zO9RyTlOI0McVVc9ytq9VVjopU2RbT85kBROzzsaH4pcx45NT4gtMaxanGsNi6YYTg5mQGx9olxAA8MlTmIA9YFZSALDDTMHT0cwUun7mN083yB46aHj0ABlJBAD7iTfSiA6p8cnTaVPc5jAbb+K2yqDXdnGclOz45uQYEtD6+JORlN4fd7AOId9PWYNcjh1qCjznmbaDbtE3aQRWlviVGIPM6LoeVppHwGEXsKwigsPoEBXodfDBDqkwTHvkFOj/gzIUk2R9RyBrrKb14hW0ErKUYV+DMJcE3WJnCeLjWeoiJBQydEOaw/G1Ugx0DjBDIVXRwixrq4U8Cx2dDlHLASqRgdKgC8OlmyQ7XIoukAEHGDgHq7JwSU5CpzD2i6WGvSzeCtW4XFBqTSgtWPyesIZvgOV8a4RrN4oiZ1epetEVM9Ee2ffG2oL3ijeQOUznEaPTujk74GbbrNlhPBnQDMz0C4G/04J4cxlVtyGiLKr78M0lHsQlepmC+eoa0gyYzig6Zr5VE+t9Qm3DuCtxul9Cam5rYNzuhdTehrJ3YrQ3QnoDCK/uYzf83bP+6AAVnBtMj7AiZ1VLPYuZ4eYGfw56iy3gY5ocJfsZJA1FM7nhYlN0gzsvlpf01kL9bRy+WIdugz0ILh1lR3xwbTxiwwZNaIl7hxenB61shjaj9G7toI0a0hJiv06Q3LbKw6QxcjiV3nltzRDO0m1qPWxy6WqlG9oiwineDFRH+xnzDK/Yy9a5ZU2nWmMTREfEt9nh8RImTKWdYr7fx/EN21YhABTR7h4uBjzkZPI4MUC76WQc8MT4HtY/zYgs+/r9cMaYOf/sO6l8rutrMr65/+O6v3DUz/Z/w270Bit3pzkxANGDRZknVfaqaXC89nmZw6EO5YoVU+6uQ43TN3XM8U3LMxx4lmfO29aYTFE9/LgPSbHDT8O9GndP88zSc7bZNxhnqPlv7JzNDjxfeA/NJf2xnTIwEiVbeVxnH771Y4IE9gQG05FUXMA6l5htpVX7FpnmCcjf0MdrQrdbluL5SHZHbq9YRnkhC5ApMEWkO5fBwZEVPJ9BkKukJmtKc71vGDnHEUix956nWnPylKmd4P5yyjEGqqNXWDCGZn604KVH6GtF26yeRs/+GhqHyuWGV4lIM/eV+fQqP4zKnEG9h1EG1K/z0E26vkN/g/lw8G2yhkPW++x5jnzwHDKUdycZoGfH8kAcgxE103N+gg4oDMMhBkuEiAb7LlFegdXk6muK2O5qtrh6gga6DppO+OiCNTKVYR9oDAdzkaE3f6RxVepg+ZR+6yWfohxlJP38tl8/mn/2lTkr67RU6dWssajyZqievJNXLqiAtEuqD7FKFmTxiFUEB95/qL9/7koT8Bik8DrKoo7wOaIfWd3UgdEcusO3s9aj+wG2Ns9Y/sFC5ASX9R1v3ntKOwZlqXOimcndyuR1bF2pTyFGwyMPjVPBxQkyM3ecESBGah9m1j3MqHn4+sOBm+y9V5sPUOWwR0zoHFfGBFWpKStWHbnwF0wLJpdUD7pE7C/L5NOjetLIemrmsuvrnmv1MstzsjbXcEVtSVcGKTRVXz7YPObJqXo+E86x3Eexf1k8NG20SjSPaba+iqc51U7/K/aqBLmHxcprJr8U61+THNBUzpsmFb2ELKe645WOGOSFua5RGBCfwBgAG6zReGaUVA3bJmlDcC2at6XnYonwC9V1WNj/3XOcZh1XOqwLI/KCDK4LP90fVPs+IAG6DL71MA/vbzRF3N2J0cTJBeuAMOWfBgaLK1XS/8k4yRd3BXAY33L9y1/A4VvC5p0lIyNscxrTjyWrrLik7ujg/bcI/3wfhNEV/Xh+evL8wgTDmVKgDiwDFxhWrloWJZCVp35FS2alH5e9+10oo5X4x25yiWfl+qi77XNlAG39XLB1IS4x9cO2Uz+e6oERmz134bkY3puSW1BIiLJKvDqHJin82bgXxm1BmUrnpzNnPDmSEgPe0AsE8rB/GDHkTBLjtt6OtANPL5W9TkqV0+yiYbBtInFi9JSPzOB9qFDhDAMbGcAVH5ZEzcgAFMnTG19WYHRw7DA6o5bXdHyTtfcgRWkVshwDWz//nxGSHBvPcp912l+cu6xzTNV5009danDKzjBkeGt31An2lfM9wl9UFqXTAZai5NLcSNKygNSB3dAYUhlxkBWnkGYYpZcsb/YMIu0uOBGpi+VX0U2StbQOQu6CvXKdXgZsBB17InOTlRe3HheQ/bprbzp7MS4mjTLF+UAR8mOS3LgNxXliDd0BS6FzEwtDgL/ySoYfO+se3cVVFnZLngy7/ujJr/x7VlI43cUkzRUaEjAdht6ukrt5E+Jek6MY3INbWvJAQ3hrCVub2taRx1FGtSbEC5j2E+iUY3Yk2gzIDB+p7kx1wFwZ494LnjWjWIy8iavA2D0UH5x3k92scswH7J+W4ED4HyEEHMGolZTdlm04bhdltyP7gNuwMiBlNA+kcHEJhma2E6svYEMvxWDk/wAu3rx/\", \"models/media_utils.py\": \"eNq9G2tz28bxO3/FhfkCxiD0iOxxmNCtRpZtdaxHZcluq9FgjsBRRIRX8CCJePzfu3sP3B0ISmzSlpOIxD129/a9e3CU5FlRkawcROJXFSVM/Z7Rkr06Uk9lPcuLLGClXrsoGA2j9EEN1EUcRzOvYL/VrKwG8yJLSE6rBQwSueQKHsVE1eSwVY0fp41LTmgc01nMXPIxKiuXXOZVlKU0dslNncdsoPAEy0P1M62TvCG0JGnekpUVwWIwGJydH78/9U//cfOJTIkz9PL0YeiSofdrrr6Z+LFis5z/mCX5cDT4fPb29NLYl+RHfDbJluL7UXzTZaS2J2LiaAnbB36YrdI4o6EfZ8FjOSFhFFR3ZVW4mmPeR5i6B/hfv3XX+w81LUKYshc7AHkQsjnxH1hlb3EeWTMhgGBExm862yYDAp9VVC1ILx4xj59oTgAOSbOKRGl3sV6Gn87kHey77yNYrS9YVRdp7zZ5qKj0o4Q+MH8excxBndEHmmVZLPBLOKhDfM3IK+v5PFp7cbZihTNCwrXUJeiEPjKfLVnqh1FSOqsoROBRCvq1YNHDouIPHBPXsjs+BX/uBdLhcHialnXBSJ6VURUtGUFoINWEpSXoZ0nmWUFAx9eHr472mnp5dLifE5YGGecF7OdwOAkrYFNC186hSxwHcDhFVqehIGo0Ii/IwYjs7ZHDEfkB/uh9i/594gBbNkpuCbyuhCOZQqssiQItEM72JCqKrACFRetDhb13SQiG7At5INtdkkSpP2sqVnK2AVkH+4dH+5x9uKDl2VsJmiRZyGKy4pSWpKRzFjdgo4SSiqHB0qIhiJ7QNJR0gRtogPw8pgHjFEQpRU/Q8hL1B1D3mAJQ7bQ0gycqs3jJnNFopO0gbs1Cqr1ez9Zw8hIUCWnRw2VFKweUrfLL6HdG3kwNLliGIVne7hy0sxpYTguWVuBFwqhwxEM5vSlq8HocvZ898kdtPlWS861wYg0Gj+IL9TcOLAZAHYZexZ2ZghFTWMJQvpPWrd6drgPGf6P1XmQp0/SiSoM7R4tSamGdtCoae0DyUtHasnJzlXkkr07jKDV9hWblb0CUHVG8a/HtwDBaLw1ZUU6/Dm9LVoyPH4CTwwkZnme/RxBH9l56+8T5EqWgICW5uAFF9fZ/JjDw6uhnsn51NBp+20TLNaSDFR6znKUOPLs8QGZ1NT04BKWHuAMqlruEz6tDQSxYzYZ8Flb6834WrBao9Cjp/nn8BIs6RUVHJB46VwetDU385avR1l0gBvTifPN22PiZAczHrSs48d6qiCrmcGCjnSTOjacd3dl2WqylJ03fYGir4f2n3rC77gKt/sDN6zpFKZ7iozNvPRULhR+qsoyUCfggwvOUryD+b0MbL+OGQ1r7QVH3iNHCyqzpObg08HJ/yob6THB3AzPOcfmJ82I7tJxC1qfDOY1K1uXiOwq8C9GxK4fc+nwY+6pdVEoT9m1CvmrmIHdFWEIz8pdRyDI/oDnIlEGwy2sVgVRGIOI0ZIHeZ1x7Ipa6ZA5oRezWf3QUvwTghAMnEjjX1CWNo5BWjLy7+uQaUd3ls2h0mJKCLgDdJICwW7VhCMCAYLt0GCQLrit7pLkXlUgFC02Z9rLzNuVYgXXIE8JBCuKBdTwBMrAIniIXuWrlJVDFeeEgTgiSDtJ4cnzlX11fXvlw0BEBnQR/aBGYIn2gmGjvAISvQWC/THGpenpDIN57+5p+ge9HGONDVVbR2OfsKmXWss8lsYWY6+PzU//k8vbiRhDVhuqQR7xnNn45e3vzQW7k+0RK9PzGD6dn7z/cmDuBCwIpnhfHJSh81KdFkAWLGdRGZoa7KcPhWcpVS6qckS6GrGJBxUJvaCVqANlFZroWC11BlEpWNw2F++fCKejKz2o7VwNgE2UUz+e9rR5/4RC3mpCxRoRLZUGhSIx55gtz+sSdHNg1c9rN5FzRJ5gzz+oiCEwzE6h9MeH8MIT6bCk5KXixuZinhSaHRq4ELDnu2JSNLLsQUHe3Xe0K0ZYi4A4EPqEFije0Il8Fym9r8WPxraMOKymF3vydTwoNgdy9csTiCdmQoVgye4A5MO80pEVBodKuaIG580oqhHxcaJXAbLAV+zXjsRvZwRjGyTKT/jChVbAAOzd1wtB0tqZBFTet/EG2WAG1RHnlgubsbnJ431og+W7aUseN0BhYGGl7CqiU9z27uDm99o+vT48RhLMCH2WDaJ8XI8JikJje9fH44uRfl5+OtDtTxEngBT+80w6DsijgmnEjV1KUxbxSmYonUytlItXCUTFPqISfF2wZsZU/e3XkbBGaWqJNGUhUxVeb1GPdNtmZ262v67g5qYS8KOAZgMRtFKK6DLUIg+x0QfbIStVc1aJOZtt5ae3VZ9zkqC1oATt7dMmsnkvoqHfIS0f1eDhql9zxUIDbz79cn92c+n+7On3v//32+OPZzT8Bz+v9e8ves8ftfJDPojXmgbAkSqBi5IVMoK+r+fh1m9SwJAKnU2QPcPrS4VB+zWbuQCtbFK6FIW4EUGMYEumi8tHPKJ9uCYavN2o7rTiquBPredqlNhnQCA/yNkwuAPDufskCvewAl3Gd4yOCVyymeclCFfC9/ZeiVPLwD9QEY4N+weygLsCJi1xFcQH0RgJS8rCyiTe2hiYbWYa1eqzh6kDNKsrrqXbvnkGwIkkvzwM8MVQsUHghe9RiUFyTaBMvdmD4Yqn/6G8mHfwqV9IocIQPgUpNbGNWtTnOphlatMFYoRqLrI5DHwpg7FhqyKZqkKjkyt2Cwg9muI5dl7SnmoKkrSnwEk5XHByCtcdixcZ+pH+8qYNYFvYpnQYgfrVmqk+sWQucwzJr051afOi0RCwqpsjfgbJQr84xq9GmC5Jy9WG7aZpSHhdl7CI1VjjfwCa9w5IV0byRuRykJ5DUOzqH6zRAIY5+5ushsmKvm6nsUlSKrhByOmZJXjWigMFMsSQBJKxp3Ih8DawY81dwdCnrFDhtpJau0C7sQYJ9Nf0vvAe44TPfUdD8p+okTMp4MiYDhVnPPlMz9aKRg1JGiJHKrslGKxpZ6sBvrb+mfehRTzQtVImwUbXb1YBs0Bd16icZpH5ZwUJ/Pk9y9uAESWg1VkHBDDs/Tls33e1E63iOmVgNwnz3DiEaFzKylyRx4q0KQAfa0oDFInpywYN2kTzKGYbMOSZqwBpshrZCR3BAhQbsXfHGFtCOdITY9zIm355+vrj9+JFPQSFvTl2dXZ2CgbB1ZfQyLQFva3+B4JH4rjQgy6IV6AtGTjKMSl+ejoUQ57kK9HRJtnZIkEzvMYrjLX0RPh9kSVKnUQA+wGm7fqpkfqobtAUn9lA2Glc9dSNoeyGE1p6RzBpSl5BDdvpQvQf0lUDIdPtB9r3Dl31NV24dfJf4jc4Ugp0ECA5g2N8KM2R/I1CcrvMI9H+TviCD06a1sNntvANF4HRAAhjj1U5p1CRPnv8p4e4u2OeFagmUC1I6AO4lwG9Ipy5ctQizsgaVBbrwtHaDy9WOprsSogfowJL5tA6jbMKdGMiY3x2ooNXvU2S+Zl3Q8O9rNhYBEymTsQTq1g+eeZV1fnUkAgecJI6CCLwmEkAWYJgxehveRsO4hDzFexoO+0qSW6pTQ7whc6iNsfNWZiK9T1gYYfNNRD5l9nFWVgLICQauktQ5IGmA5QXW05o9JkhQTtPledZBZTyRlFghTY11otrUrod6GzzGAQqWgom0ARkOkkRlidwBFDwee6pBh8WDDz4VhHPXIhiKOIEXyeOG/8XbZaNvMRIyvhcweKBhaxbU2Jy3Ysv2AGL7101zKngubZgyRDLh/GXDVCYo8rqq4947TgSA2T4EB/r9iLof3hY4ebScwv+Sgd+TD7QIVxSEfvH59OKE0ADErrwm/Pf+6pYUQlYlcU6ghJyRmyNs+mDaI69q4SGOZzR4xB0nV7cctDRZ0aZRV6Eta9Ge7u51bVIECy+oQwo5ik+XoIeYSZl5ig3OozmE09C5G46DCX+JYAF0+OkSFnCRcwuv8Kd452Ac/IZfh/tiNlr784RPS9McysL1OTTyxB0cYLINMkRgKub4ffDTE6g2o3hZBzwBmXbSMJ7ttlFI2mA7AfL2udOZkvGBfee4xFPgrWNHEN0coeMLN4NMEgKKvAEUrbm9MMzN/AhT037Y3bIqofw1kf3JciIEIgcOJnSy/5fh5rZ7QCkOtB11ki3nMX0oEdALlAavk4fbaAgmFFfi0TgJUAhBGVBu24D+o3WWox4K+1IAXxhsKz7Tz3Cm9t4DtkL9znKc5ud7CKcyghwfn4AmgWvnW1pLXC2ghi2xM8t4nkCjFLulPNw4zHvwILzV5d7nrJhF5d7VyfkIfT569hluyCNwwOD0MoxZvSTgESgNdlGL3VXjD6rHbiryB9TEUhU4LadmJh4Pfjp83EVzdtGefg3aRYuAqE66ZXVJTGmlmRD/Toa8IQOa/j/s8r9vZvLUo8Fgm5lhfqIzr04HQROzyVTtszF2D55/J6BfNl1Prn72L2xPKX70OZCW5j9z762B/A8uvmUSKRlo7xLn8sFjYU5unBlyniLKndHd+Gh/f2JrQv/1kSjvk3qN2eNcXCY5wJNKXporTn8bTchXA297A9zLJdSdTvY76alHV/3Me4phnbcDRDLXCuL5Mu85wfcK/BlBPyfg3uLtryKji9I5ZPJQSfj47hoA5m8o4qBfoTBEy88ZWLdF1lWN6MqC+GLGt/jzdNK+3Xpn9uZdYj6IfQGNWft2nbwTQCBYlqjxlweHxkxOw3bDK1nsabidkg9bSOKlvPagkEAXGbiEDHxJDNkjKh4C5u9u0GUWheTz9fE5ubw8x2Q5xlsvcfRSlFnHdZUlkH6rV/dA5WEzLxr5qzQQxBEewSOUPN++fXuMRd84m48TlmRFQ/i7IKVdtj1zhYU9V5ZWnMe8Ly65JLSxr8XUZxb87ssChW511R2dbGuX2IK2bvi6r1PBefALz4QXZVzUeET5s7seS+IpSvJ3BvJxHBOES36EgiasmpxNYUUN4n/dwchF6K/xUmAFoc464li+MWoO9mzGBNpZ7LTZvkeAXL7BPB5yvAdss3BgPR4HF667C9dbXH82F8dZA8O2Em4uRvKb7uLe1VHqixuqtbxOErjGrYG5ZH+0bSdUW2JflMp9HYa9MMCsRk8T0BgENP8BAY1BQPMUAYtRPwv4iig17e1OUzVRaFyDV2pwfb8dYsaVuGMkEtcWSlDBtTQER/tMxFz/nAyQ8c8C0RIQXNwF6VN8R2a3QPoLkiLLjbO2aqd5/DTVfL86uwXsBXE0X8YmT0fPUdIoShqTkmZHShqbksakpLEoabbLH1zfnbFw0gJwzZO0w+t75f9hZHshZZI10QS7Fuf0xHqHkkcGAUFyN18yU7uel0Qh+Ax5F3NORBAcoifEaoKN2n/UgMHIsuU35ODwdf+bo893pCzz1Mt5h9IPaLBgPSlzJ85aj/ivDp7u8XfSrH8DMPNEqg==\", \"models/upscaler.py\": \"eNq1Wvty00qa/99P0eupLSSOrNjGUEwKU5sDgaU2hzABzm4tRanaUtvWRJZ0dIltpniOeaB5sfl93S2pdXHImTpDAYn68t3vUrhLk6xghdil6zASo1A/bzPBgzDe1AvhTozWWbJjKS+2UbhieuMDHtVGcUxxoVq/iI8Oex36hcM+lSkgV5D8u3n1a1zu0iPjOYvTGk+S+duRArhLAhHl7k4EIffKIozyCrg1YvjDi2QX+l6Q7OMo4YFHDDhyR+zCwkuzZJOJPNdLsQ9wWBR3odh7q2cLtb4OYx6F34R3FwYi8cSBEKitMF6LzANeAdgZ32ngYe6FO74RBr4dvxWeuBOxF4Q7jTBJ8aiA+jwtykz01vdZWIhMLcvfFRoABgX2aPT68td3ry7Zko39MuBjFq6VfFx6dEEHv+NhxFeRsGwGUQkcTMvx6M2H2TPcOnl2dB+gc0mO2l5x/1bEQU7n4thdQYjbHc9uAfxTVkKnnz98fHVxdXnj/XL9+vLqI9b/Jq+P54dJXgbJ5HNUZPxVsku5X4zP9a48EYNTrAyeZJZ8mrzhOVkSbI/N/88eO83tMovo8rYo0vz87CxZ/VX4RQ6qoRi3zCfpNhFxeJjMXCz5kfCjpAxcP9mdxWf88Oy4Wd+ld99u+dnqjBQykbYWrCak0/wsORuiyk2LrUlD7vOIWJirte/qx/hG8Ghy+fHm7cV7Rfsw371j7O4JuGSVsE4zuwmLbbmSzBzCOC54cmYAO8tEJHgOLirHOLubunP3qTvFFijOswknfNIG7578C1y9FbHIePRjvn5J4Gz4H7IFa/9WljaKpslh8RCWvo9Gv15cvXvtfbq4eXv5yfvLZzx9endJFozY803EuSisypb/8XcNDcZa/bZoFhf14kVZJMx6z4vwTtjtRQBhnY3Z9Pk0Zdab/35dL83/h1mzxWKa1isLrMxnz9TKdwQFr0wlP16U+OSIdah2r7AA5/ZIQAhZKnyeyyD8JS8yh2LyV3JRcD8KxJpB3GmJqFPwbCMKil3gO0ziXMXXPPO9/TnCoI6H9Lw1nvW130oE0OJ4zgiHCoeE2bsV1ZrNJi9VFvhClwnCVxVmxuOx/HmdhQJaL4B8wvccsRJpIYVLTxCwc5HdUWZR+BgWkqikoyxNotA/uhLEJ7mbs3yLAD4RwUawrQg3WyxZJOkzkusZidJm+y3cnPkR38mUFSX4T954sWSLJ/OpSi9xgKyxRiCkMyp/ZBOSGehEqGQU8FkjNLfFEUKslB+BnLIkU9KTT+e1ZWY8RND+lUeluMyyJLPW43fxHcQZsDwpM18Y4OFsEuD3g/y5/T62K0S1vFmcFJAu64TlexF+jvMypaQnAqZNK1MQgbGGTNgklFjasCfPwZagSquD7Ut96esX7XpfFam/4YLVthqSTMdtbDdK9iKzNELwJ52N+PpNHoe/yYeGrT8xnCBf2CeMHC3lOaIFs3ZlVIT0YLuGCPbQylJr5zFbtHa2emdb74iIKJi3KJgPUPAh4lgi7Dza82POgqSEleRwUKG16cB2NzwLsJqzROtNS5RJSZ0mc36SzLlBJockDToV6C6tXdCmSk9i6R2ieqOBScjJywzk623QxWxEDemmgE+X6n0t61sDCnntQ8DgWAfMwgRDfv8AMHTMAGNyOHB6OjL5b222/fzBUn+Q5Idpk3i9HEkdd3dhbElEjgJjt45SvKtP8sM9Jys3b/F2xtZIMYXVYGxfAkhPhlQVTN1paxeSavA/1hhe1pfaTJk01GAr/DUYe1DQFJqypIwDqxK5hGTbg9Jun94ap+XxTKCCjzuFviVROQqGrZMqyamKoyoHW+18SPX8YHmsS+9/X0Bnj4yA/sitQvoexRdrlRUtv26IASGd4qItSiWj9hEjGzgPzxY13J0oOAR2+lJ9cq+SPRRDEqa21Kq6Whc6od+xZdk27GfcVlA+HoLi7m7pQopqJC7yJTU8DhOHMC+85FY+GnRW96g7BnqTGJjruJG7m4k04r6wHrFHDnvkPbK/y3K1hjTU1VpfSA5fZNX8FXI0sTnk7N7qWIh8+XTqTadTg6wiO7Z1JBvrPEVtg5q6aqdlgX5FWstGnUBBa+DHOGF1nF0a1lIfdRXJQKLohrlbJrH2wGW3SCzV6drdQEGNbD8gqFtbHq2HaHEFCihjQxx8kRbsUv6gopHnTHRDMznRTRnTlKNyozecGn80wpK1ez0JhZJAgaSEKxoRnvQEimb0MOq4jlxUbqIU3jjE6L9USy6HEihGhYQKPinm1M5LwwpLQ6HfvdUmO2dx6sYBzzJ+dLQ3y9OylKekJ+vz5pAOQ7uNl21W2Pfv5q5/V7xKIgimgurI5VfXV9c33s9vb+Y3b3+263tFPX2QpiDnPJYGaLupyHZoO6y5w6YOm9muCue2W8b5b6UQ34Q1JS+dP32qc0cFU/7s2kvPTtqntZ2o2UtZyB0VluW+3vkT+98tqvkyp1p/cWAcxIcFGpAyE1rlFuKIy4ZaaGgeNQan7l33KFK+joZMrqyS2NR9etA41jTaWIVRGAuekSJEhm5GdkAo+jNgjmTAlbMca+4+OcgrMMBiy2Pc9Ev8Q5NShCikeV7LAuh5AbfTVqBNyGHvk1jYbImcLDsbww5ocd6IrxKSUmAcu+sy9okuHrkNncJquZC85PRrB2+NzJZkS3Du9Hx1Oa4EMG5vgqNN7PlJhp4+X77hKHaaA4Yu4xR0StRuYzm1fc0cRibW2Jds+Cxlc35aYkVZpm2m+Ja1WxrLY2WMtgsVHFNhwVlKyOK5bboBXGAOV6gKAZ6m0dHzszBPPYAhRi013+s5JcKkiDcFmmtJKpWW7vOng26J/vK1EClslQDCTgupTo0gV8kcBkLt7GTH81uqAuJ8y7OU0SODfUUlUgZHDHGVKD/SrojJLJMYZkcdyxXzYWcx7J7Aqxgux7uw2iNCIpSrWmaau2yO7BZE5SCfF6p5hp/k6LAFj8FewhRx5BoBUZ9JOkG0xLUnV2Nw1CSHF63LKGKQZUW2nrCCdEC7A5SV2IZx0G63I96LVLWsu6Hq6kKHqsgjHil58dWXc4fh7/SroWKpjCdzbXCrqMw0jre8zPOQxz9jyVJQHEaGNYVBzNyFAh/AE0OZGxWeiQShmidILqduuLZqoOOr3CLoH5MVohOd1ZT/6j2ZvwFgGS9v8/CbWD4xMulPD7k8lffbl+1GtEuC4UdhavX4k7Sa7M0pOD9DXY8F+m/mViVHI1D1y0+VDB7juvsMz5WlY4Xw2jpEapPmEc2yjpOAZszSvtPwgNQJqyiS0t/CNjIYDFhbwXRiFOIZ87My37It7DOSdYbbpgRs7XFDaDWxF+zPz4joWicWPeOnWrBB2NSdPdf0Vecq/gw7MURWWQDAIkz0g8TJ8AJ4pnHCMM0AUiV1achWM1lzeoMYYyg3HGHMed2+P8Izp3qtgHXOVklCJqxj8HClsO5cGjXVpiYGEO4Nh1q+svVydIu8bK67CASp+HI+/2pSbTSG1VRpD8s0OtoZ7LVi0djZ0k6d+C85DEuleBoVMTU0ikKKNMfWdAZ5+GC77A2S/eIff0ffV8YyftHQRwEo9qEv+nMnPY+iiYEWWGuM9pLNZJRt8fWS/KoRpZ/A2BHLl7TctB1qkKn3XrQhTNhMTJ60S93q6OPl6QGEpvYnoBp11tRkQT3A2I1UrJRca0yZEkcSLwN+77soZbUQqUflTsbjjdAIbIP7MiOt0Y+9Tvl9s1DqvI7Z2w+fVR4kpWSoVcyZsfViOZs+f57SWIi6HxT/NK6hOdRh9uc5hTguu8VC1WiUjiaSL1bX3mZ7XDNJGrQkoQgeitIXECFA4plQEkaL5GdyY8tDtCurMlhof5eosjuTqG5j19bDYEOgt1s9wNK0ArsH0U9QX8Zl2z50R2UZOr0ui+v1L2KXZEfZPzmtbsoe6LmqoSHVADQKlXfH1byD+kZRD4KlaOQOSue8oMrFQi9+D367j6xu9IbI+MHb0O4f4zgZCsIaYkhtzMoOETEjen+KuEPJaZLRKFi2lOIgfG2LspuinAwTmC+oNiPbJW9AD7IWeyr36Y2kw57O5rT96sPnRkvq9jc5RcVt0xjl22DcGfVto/dau1/Jk5W0BwF8twJUunR+utu8x7IGWoPWgfZ+LZalZnFgO+XBcvas1xbAc05HCEhoz/6jHmfuySO3xsLWaCJlp6PrvXfvP13eeBc3lxcEokk2L5gEUaeYF2yrX8Q3t64u3r/6/+uPiwFN0CGYBfi0at+sgTeZy3ba/eFSPbXKCn1fVw5plvgiz2tF/TVZWfjnIIxD4b13gjKrU3+o83mcApqeZOGa2ywYr/f8bUK5bqlh0ozNGsstNJsnBpwDrw07ANqbBKnzcqhfowACFSmWCaZ1AFBk+WI36Yr4+X2DQnVJTwobCA8eErYCNhUACGckW4SzTH6rQgpDg9UNOXSGPrlAiEHYi+C0naGXMoBRawbmdF/TnRxKKy3aJmVbnrcGCPQCth5Ot0dtcssMe7je+jzGakynAyDcbbQHhDsq+s2T3TEgnQ1zw0IHBbSWAztr/Dmm0E2DO4KrrJlJktxxP8N1xDdQhQL7QKFRefJe5VVyldOv080XLE7HARw2rIl2qGj3AgNKdohOp0WS0/aTPmjt4oY/0Kxaowq8v5Fo8c8Lg+9uGm/GXb2QDSsNyq+X5LS3gmvX9YZNIZKOVluu9I98KLn2lNlMYSUKTevvUifBLNMAZWD9SZg1m6rOVf2tO9n+R2JVYO4MrhuvhM4jUYDUFvODLtrMhav4rD4D0wK3uhgcIxI7hpYcNqh/04z6RtYyBSNpPmBEDk3LLJA3Meh3xKj+m8quklWxV3W99wmHhNJJUcZnJ42EzmVYdx7YMpuCM8Cd/Lzlxx1yE6l8TjUEPTdf+WXmCr07pBq454a00fLAXboYN58sDl8a9t36Ji0SXFlAyO8qySlUFCAtaGKGUhYYcdg6zansLoBeBqJcR0kd2oie3lePg4G9X+T84fFTf1w5jKr3EWbFesVjvwhr3FelbOpuqLHFD5f+M0xedcFhcKg7flVD50UdWvRtejneaeapZjjvRlpS3B6NqheuIckyJw+7J3qeqhWGgpFecxTVpAaeujIj20PxHkf7iFc4fzv611OX+rp2QOr3prDOJ7NWpfG6lh7SYvtlaKUnOevosmuoEduUxZqV/2QLWpVfdlmmR7CXWKR+tXXbPDGguCHTaH3DbA22oTJH1Hh6nlmbqSGSHqZlb6WHql3ftfia3h/Zx+9k8SUdTc4SOJQYsG8iU8WZLNQUvd1kbiRyGe+idvlMcRUlIVlkvyxUFiw/GbVapa0Ovycvqn3j7uiH+a83iMFpHUvuqXOqE2UMs77tv62+/ijHGOe9udzoQb5u+niLwMGv3ZvYZ9YbdZZB16i+wRQeL4Mw0U0OCFjinz36QU1Uw7H/mGJjcPL1IKHfK/gfCb+ngD+w8PknJFziUA==\", \"models/denoiser.py\": \"eNq9Wetv27oV/66/gjAwXGpT5MfSogvmi/U2aVEgbYp0t1+KQqAl2uaNRGkUZccp+r/vHJJ6226LO6xoGumQPO/Hj6rIilxponlWrEXKPeHet4qzRMhNQxAZ99Yqz0jB9DYVK+IWPsCrXdCHAg7U9JfyEJBrEWuvZhHvFvWjrLLiQFhJZNEIyFW87b2EUpot0rP8szzhaRlmPBEsqrRIy1oW9Qj8YTrPRBwl+V6mOUsitCcwKzwTOipUvlG8LB1JxsAOiHwn+D5aPb+09LWQLBVPPNqJhOcRf0QBdknINVcRyOXAW7HMMRdlJDK24R15ecGlYxCzQldqTN8robmyZPNsWQITkOZ73vXNp7evbsjSuSIBNWNOJ3GVsAkRa0fG1xAUYDsmUrZKOfUJ+IiTSVxUE997/WH+vOFxZLPnvXt7f393/xE2fTa6TLZaF+XVdLqtNhsI55rFPIzzqayUOOT5FE0wgUhWF5lQKldTcGqe7vg0Y0JObZCm88eLlyBT7Li6eCm1uBWSd6ii1FFDDgu9nQR96Ruht9UKBGfT23zDJNuBt6bN4YsPKv+Dx/rimstclByVSDkrQUYd/uluHs7OS/zieZ9e3r69jq5v3t+9/XgTvbu7vrlFX0C+PXFZck2/TuaPEXOmRAw4pMhh8g1858UpK0vy8f7TmzfvuX6VZwWLNZUyfJcnVcr9K2NUwtckioSEHIxoydN1gPkPhCjeLv9uX/JKd97WnOnl5Qv7Eudyt5w/D0hVlDFL+XIeEBATQbnx5QQSOIVIW0n4p6wKrqgfNhL9dglkhzVLsLJ+7G9Y5ckBF2srbsF1Qya4J2QF5EKC5r4CJgt4qq1qrQgImAQKz/3zHD7c89vfDYOCYR1AtMtlzcTHhK9NJsslcVbbVIfT5rCQRQrJuvy3qnhH2jpXJILaJYrJDaeN+Rdk0fHaD1hmzTln2P/fuB9W2eZXV+kmNcFBe6YSl5mPHafAIUiEx54rU3bgCt3ZiO470Z4xuyg8t6oqDn1Q4joUTpTYuo3SPH7ABlWPm/AWCJBtzQbTT2DH+1xyOIgKb7juL1Pzb/TAD1ek1Ap2nyhaZ9smzVcsJX0mtpevScOLyFyjocdaRGuzYsCBfGJpxW+wG9L15HcJNYhjgyfEiVCWLfnla8P+2y/hxEVhD82O9FzS8geNBr4QpdEMHdJ3vfPwwKxmi5WcCPQPDm1az/wQHIrPsER9n0zJpMehnDQs9lxstrqMEAIAl5bjFD1+stM2549NaOomUNDjHpAMOsnqoHm5nM+i2Wzmt5ZoNUg6BwPKgslEdW1u7AZt61Vsazy9BfmmT1pFAF9YbSB/aFeRTrHxx5gXmtyYXyKXg+6RNIMWWY4ZgUmswOgyPLw0E7o1OpfpYfmaQc37Y7ZlgjGiE9M+yohnDE72iYYw6EWxHUjAYDSi/tT8OSrFOrLUDKBMArCPgjpH4+D2DxKzTqdQ59RiHz/kO5Z2Jg+UAuKZq6Nc7eEtS9e0kymjNtKW+pmK8f5l42gAHwBFuwB8zSSvNyMQtJ0nIPgcrTbqCgBtKBOmFDv45OLXzqtVW2SbSG1W6IbdIox3EJEUmkZ9PjDkV3e3d/fRb2/uF/dvfrP26ya5TKoa/EwdMz+EiZ9VmtNFQGbY3cM1BAOGdljJ8j8V50+czrCwF8+ehbOWX8fX3lEH213OqW13t01XN6RIFkCFh7CV1ugE4wbVanUC0JQV1OoJNQAUa41l50LS8w51Mv5qDfBDVuKwpODcSkj9wu+6DVy2ANf5bljU0TLoug6Xhdo/Eq+YQZUAbj4LonEjdKy9AY6Oc1huWcE/Xy2+1K5tWEEbInQL1uzJP5dk/o/FDJ7nsxcvCMxXCn2PIjPfLCIV92fssUOFI35nTo/64TCvu6naaDhqbLRj4V2l79bveJargxlrAbmvJN4BzZuP97LB9AELJ5geOYxQc25SD1DshBwb7R4brjHHrJRCQruA8qI8IGdkD2BaM3WH4k8GaHy+sxUGnz7ANS3e8rpzmGosnwj6f3HZi50BZc/mi26uju6FtEVMtbODhgTZvwJGuP3qdDPx2wM16G/DbdSDO+rSKTpYKliC/doQ6zKAy28Md99G3h/5isJPQOz0uDI39c8QqQAv7l9MLbQIA9AnVKCb+nAsbAleCy0QMy1NtC1TO57MGkynU5Cs6SLI8afQiT3j4EnLIMwecC/owKUuDWIOIMURluQPFkB7rraLGlg2t3DVpSi2j2zH66g3JesJLtRafEV/wE8kkm9hVlxO2s8Ixw+7g8mJk0hE/tDYoIPitw7Na9ejb51Sdap2Sx8SFQsLOUBhKfPJA6MOuHpYBLgnY+oBEh8KMIXUpf6RFjKa0Wewd2fsgiK9ryK0zRd/iNw2bhaKDK8A3Z3D+sa9AH7HwLdn0JphIVJA4Vj8UOkE+doUJkYlxN2jhtI3tx5sULmg3tEZAtr4Q1N+JuCF3EyGFmL0rC9MMhoM6djCjHMK+TgmTAO1K6HJ7fJYmxs55bX5jU4xAmrLkp9yDHKtisTkpfumRuezWTizd0v4i4/mffyVjdZW+CfyEdBhCjfkpGf80dxsaFDIMNeKEqeIhrIzcYK3vUjwJrE1EBvjMvosdzTd3Kc54++9/bXtn7Y76lJ0snvSOsVgSxFnJ4II+BXiP51ys0NCJI+wPuvMiVI3jnOnwact3/0WYkmwn40GMXpzz4SOxBpsq0qs7jPZcaoHHPO1ozkcgbXLitBUrn8sm2HrWPAK9j94P1Ns5qUvYPDRlHajZpB0N3z9o63D/wYj3hvq3YkHLGO1tZS/kEukzgxU62Yb+RWICG16p7s7jkTgWIx736rp6IgLXNDKGWZ9m28dZ4wkLUeUkah+Q+/ZNTsxTeo+89Z0W1MscN+TmsGsT8gTV7Ybm85s9R32nE6/Gd63B6DT5TkMmiZ/f2LGIZA7b4SFrcMPWO10T/uTF9HEyU8ztkjMF+r+hdaBjpMH7XrnrPdd08dfR9Y1kjkzKuodlQRc9kBHl4O7jwaL9w8WrCy9H3J3t430FDz6/y1tZ237c9BiKkCtUBxc7XjEqkTkDuOBAkv48b3vDJWGjwvun82zkcN/2OlnHf89548C8D/M+f8CoDqC7w==\", \"models/rife.py\": \"eNrFPGtv4ziS3/0rCA92I80oiu1OGgMfPLjuTtITXHenN/2YnTMCQZHoWBtZ0urhxB0EuB9xv/B+yVYVKZGUZCeZx20wLUsUWVUs1pvURKsszUsW0U8cXblVGcWDSLYmRcaDsn5Mi/qu2DS3JV9liyjmzfMy534YJddNQ7RSLzcZb0Z+i8TARZ6uWOaXS0AvCWEf4VG8gCEArG5/lWwc9saPY/8q5g47joLSYe+iAq6fqwyA1cCD9aS+TapVtmF+wZKsoSPNg6Xx4CaJu6iSoIzSxI+x9+lAELBKQx4X7oqHke8hd4qaGGvA4M8v01UUeGF6m8SpH3o4J4fe8FVUelmeXue8KETTIgLo0TfuraOQpx6/Q0DiVZrxRDYHflZWOe+03+ZRyXPRTPfeIvdXcI0Ahj0YHJ98PXtzwmZySiFfRwG3hkEV+kMWLWQzPrpR4flrPyI+WjaDGXI2DLJqaA9OP45fNjB6Og8G37HZ7/8DKK/94ObWz0P2i59n7DwrowB4fxqnt2yR5uzi7PSEvUf2s1dAC8w3QLb8Qei9K4E98+I0uMEJ15LrvoMGmKfqUfLkbR6FUxK4OUnavChzh9WXS0fy6zNPijS/BHD3D4PBIOQLhhAsgHCWZFU5NfrBKJ7gfM1mm+3/ZDRMacmHw2HDMQTK/ARE0b/mBwvuI2cQGvRmVYEqk0p+LgC+C2MJBk7UAoItiVgKiU2zaBoLEFHLbjWGqL22LaQvKpfM5KCgEf9A0m5YkqL9YF0eNt3wDxp/TvPoW5qUQOlMqpTRg9gQoynyQZj3x+7IYXRpiF36GZ+/gCUQc5mZU4NmJHxmTqODx11H/NYaA2z6rw28ZwAor5+EltlzBGTsd8ZPqNWEYbcZ8ZXnYsF+DxsmfyAbupDHz2XD/qOsNJ864jK/uVS2yC+tuSExjs43k75rGAzQwKzNeqEOBpLtZG4MFANtUfDtfOqw0RRmAr/TS3bArEadm0mxfVwNG15O4MfpAzGeTnaAmLRAEAQxpUE9H5QMNa8fauC2m/F8VZXcAnEAHC8aRuQczELCTl0aVfgrMFxKtiJEP6vpcAjFDC+KfHR+s+FVBILH/XzogJ8O0UR68kWahxybwaldJ16Q5gnPi9nnvOIDsbjoLD7mKfovto7ysgL53iOX6uKCxP6G53tgyEK2pz1DhyrmhfICNIIVQR5lZTHwmq64cBhTuO9pxGe4t4Yt8ODS1ABqhlH4M4AwxpWo5p1RKHdqnNl3ay8wfQLQsDaA2jhh+rpICciWaQDxfQNcnQMa/sEA2eW9Pz8+effJOz67gNcYS1l1mOZe8xLvwygHAw+yNsyjBfdEkDMUo8+/fP745fNzhqdVCRIE4zX03sXJ27NPny9+RV8o/Bf1XR+6L4dT2UTNCcQw0DKklcbX7P/+538h1IPAkf2VvXr/lVkXPEhXK56EPLSHSjyHVR7jyGVZZsX04GBZXV+DdC7ARLpBevDaL5aTvx8g3H2KI4oDCMXSeM0PVn6U0Auix4VYVIJ9cExaJ0e7iZ0cEbXvU4wcgdxTv4AwleLIP57QydEuSideDDHSI+ROIFwuOdH8JS5znwj+EyiduEjMTnIfpRSJvOBgWwow8OxvYDuicvOn0LqDzEeEdSKk9WfurzeCo/t/HqEtQYUI8+urd2fHntC6L+8+n318d3Zy8QlUDnKXbxAN8tK6B5dw6LAfH2yjN+hoq98wSiC9yNLYByly2LAA3wKWgUQZBg884snWaBmzHx6KTn4IKQy4AhkxU5hcp23zeSsENp4gWvXLdjhtxNNNpgVeAzIo4HhQElaLrJh3wzdTDFxhjGZxKKaGVmGEwU43nWtL3WO6VLCa+xG43K9+XPGTPE9zazH8khRVhvkbD3UntXffQH7Yc4fSfa946QNBPTjmTfdL6hnyovTAvhq9hTE+UETXs6h7QxgGWlJYtiIZnWeepuDYPWBrhP4UZpkW4DviG6seaJsBOTowTBgSXrrZDbgxGEJjp53ITwYXiGJgUO6ubtA/ZH7Ok1JEAw4j+rz0hh5FdAKy7GHe3zvTxVDjI4n91mzbmiN356RklyA6NVyHraLEu9qUvJgdjbzRaCQXo8w3ajqUyshahPvfUXaKEDHvqcGAn/ML9m1hcuDbwpWyB2JN/RuO2irVjzdGWlSD7FmtDl0NonpMlUAUhpqmv+V3Ac9Kdv6JpLI7OvOLQkz62eLwRFHoiIFQlYsqwcpPrSzHcsFAV3zM5decCDKUhYUQ4qIyBhjcIzaFnRSJ1L9ReVH7ECmvlfu3MgqZYpFoewZ9cpfFURCV8YZVyW3uZ2zNwUpEQNjhMbNeU/j8s8N+sZmAx6gKdHb6gZds7eeRDzLdpNNNUSUqupTYFNeqZzcJoxWbzdhh+4VIAcaX+PKFZnQka5uONc4IEvyi9BPIBNVLh1klViYchg7T1lZyxfNrjtmD6oz5BgCKeWIQ/BPgF8Ugrev++FIXCg25AAyIYyrBEXpbTBshi9cAFdInQ2YC6BGF4GaAJtHJQNJF1AzYiqvpYSMTJ11N0HGq3pCpQiKlHseXMgFrE9NeZg0fktA8thZZtW9Z49ZaN/0HO3DXfEUE4r6FVTbuQCnRiY7b1fZLAuk9D1ouTioGGL2KKnIQIGH6okuS63kYKXneQ6O4kDp4WJNa+bEIEgpI2rl1G4XlcgrGBRZ1yaPrZUkPpMEUDDSq+6oq0/2QYxmQZXka8ILqXASGYUocMgjBKXKqKBovU+jH1+CF2Jsvx6/Y+fl77LEEJPvQjVFNt9HlLLoDycfcECli30tqap2Tr39iL348HMHbyfjlqKOqIwjRRe031odMjl7ikPHhYd+QIz1XH4PoCX5dVVEswymYYVzHVCLKIfY0EZXruq2IqWHamajh1xk0pOmiVMiVHUSJkaEMCzewbBBsa1xE/mKcV/MJYACT5N6A20AUdLlg1bFAKbwUhAD+ClmKOmtBV5daOIaG7hqDKfCBsmtaRLL4PmOqBJORm8jQ9whgul5k7k0EtMM7qybnYw3f/Xj+6ezz2fmHV++88w/vfnXY7i4X3n+d/PrL+cWxrLwo+kmQcRL3mYu3bYoehNtLqpV3hcExdj1C8pZ+4ZelXDCIp+nt4VBW2w/roAlcGqoGibEnGIW3UyH9tNC4wzEXkbGSH1ncmv+IVcBDvEyaoiBcyL7rRCFVhLlnxGVbKudXGG36d9bIHTtCxWya+BVOHFHLCpp0Imr5yIlMSKhUI5hZwTtcq2G0uh5hgnFH15KuZ3Qlnz4aar6L+FPepp6fX1vlqFM/H3dawHwVJc8k/4BDoGFyCqqNKm29MUL9hwXDayOBgbhCJiLtiLnGOWykQghN18wLqHM1gmo/8qEDV8nE0yFrY6g41Sdbdj+mZyIh+HTX51r64zTSBVhHXDmHff+9AGjbbfmrl1xTEogKA7/8f5WDO6MwLMm2HQhUVzOt1vwUUXm6mDxNRJ4jHr9NNJ4oFo+JBE0cgosEfMgTJ173RmCnPpiswTNk684QK921ahKkhSSq/vl4/eB3lDD+TXUHsVHXFG6M7E5Rght1veWbXsXu66lXMYzdg1jWMp5QtVGih+GOSFFnmKGidGJFoYGnemabpuN8QQ5qgbNRAGCekKknYYGcsIZuthmKwNkaUk4n0loGA+tkV2tB9opH20iBxFDv5+P1CwRoYGzXU6gVZckc0/QSkeJm9+D6/Vzz1OjLWwvUF8J/SIWU+NoWutxgAX5VIn66b1j70IgOxXoQMWGwZxwPcbHVw7SYiELB8tHNm7uWC6E8iNZTounmPItxB3PP3XPYnrdnP2h1UjrvULhU7vhHGiVqxZ2aGdremmGqiNKoYB/ShOPa4bNLkprXzU9g1pcE1RsTBxxq8oyWYu9e0iF0TJP0LpsEkwWjkBwLL7bBW0mhy++Ao6I/zllbAZKaN7FfoIiDvaojSkcKFERNODeDF9qYp09dVDgCGoWWqZGN1pRN7QaiFDatNnVLuVMtvtvW1KgvaZyBWXLoEZSN/0U+WRpQB4LTrJG8GR1dcSTWwkuTeDMjz6GABjH3E6qC3N80UjgUPHcx9oSok63JhNw4cIN7dw0dLkjBCuan/DnZspwMj0h8qEGNsCQ+OkMBz21yZNxcg3FXEWWzqCMFVkyopiFA2OzgoF3R6FnCTrC2EJsUkikMMKz8Mlh2C28YOqXA0GTDJBkMyWDW/XYSH2x3uGUjX/CDQ35n2W6ZWuJYkjF1PGNkTkgMWvrxwtJkTLoXYPLubLgZ8JhzAkiytR1T1M2DwX/Kekuy4DlPAhEdAFUUhSZYjY1yr0w93AsHVCL6KAT/MbXxrq7zKUsgWwz9PPc3DraO260DiijEWaJd0YToQeUR43Kp3QpWLkEByAoJEmT5ZzqRuSzW2S1riccMSKJeHrIf8PZ7uBM9bqnH7Y4eYvcfu0EonGHPW4fhLYJd1uHW9dUIugTriRusyzdpDMJZE+VQ85vzd5Bwv357Mbl4+9quB417Bo13DipHjYEgK0tH/SzE70KAAfl+WnBrQhSObV0W3Sop/llx/o1bI9ulgM2iwt/RkSz9leMtoMe/C3RTTRKcxOLcT2yEDqtueXFp1ku1ORpnFMGihpSRyIGOPKyBpi0KcAtPaYU2mR4Y40dhdDRWEDWq9dXEM9abpXbVGV+DCIW1rugLDUKVgsW2Omfk0jy69uoSIT3cqhKhUihBXZCnWUZmvsRDN/ifGF/f3F7W8ga6TLVgMcLtrpoL3nCVWbTEWPHDFbRd3FDCd1kFVyEWtiYTYzqDMzJmb0i1wKzLNMjzBOTafsT2XPOE5+hfaJ+Wipf1EVBpfcjqLZLprsqgs91O1W/G/W9WVVxGIBg8F2shqoGdfNoZqJqVAqEqkvT7Vs6F8TtIBOINsxR0YX/0STIxSeGT0ZvSseM092NGJ5AIpDj1mPOgygvcXwo3PjhiVlRXYbSOCixlUjbkV2UKjhBeNUXhBbALz4i5Bo2YIimiZLbWv+H+zIRNY+S9enioo6taWbxa9muxx6R9uwNSNrZeQ1tWTzLaZL2fTPFY3+EUBfTHKXvxMFfIL3u2RrdVeqWYKb3HMqOq9ULMYJZ6VcelX3hraIfkHD1xsjGjlp0F3a+vLjxVse2r52IPo4qrb4VTxXZLCbqVUshNVUutIrjhTSZutdqkMe3DLZOkw3BaFQuDF5RUiPT9tqW76hSxCqlaEEbqBeH+chUWfEFKdfyY6epkdrbG6j1GsaSWD0TYzwP44lGAOA/NSbR3ubeMgCmrQXJJmlV4Is4W31H/vVrZn8d+sCpepy7fvwzbJmiufTPLBnJnttoR/POqPF+856s03/TMf9fZ/umWs8TUFaxouQG6giVvHSzIeVgF5EfFLsDkSKMUnOHIPbKfMUkJTl8RYamfogUxX/NY+v/edSBf01NuI69F52Z715+oo6pwzxpQgrbGDBe8Wu96zgF0Z+PENGhqkngYJZRTQQfXixP/fmjBFW0NIIJy9Rig+tRRFKIJagbXjoU8Qr2towcq81Ystmp7ILEJtEJbisAvZQgXpKusgmBEOw0qSh7CkSbeIlOq1I4jKOikkqueE8nOV2kat2KHTxyND0QFSZqv/HgfEIGg0q7u6cdPTJ1iQ5dPpzfwJNu+OMnG8KSs8vUXNG9IdYFoJNLBHeOC52vIHqswSu1/f0igVWv7MOI5vufXi6ei5NeEHDUCEHbj3J+C/B15MmIlEy+n7IbzrGBFWuUBZwEk3hCxwiMLq1xwP0qCnPsFhG5XG22O/8HwuLoqSeCpIOS2KNLEm7ZSiXhciJHtaHsC37EPugwYSz9VCAuGgoGuS1BMQaei8q8C+2ALRrB2inRALzw6yb08hiDk/h/plQX/HLkv3N4PItlWxTg6fV+ff4NhrmpoRdsUg5WWgIonsK2hejmEdENun6l6/ow+3DEG1O+w0qXtaqiRWwZhf/1cqK0JzLM1Aae58vMbsMFRzMPfoAhqiX7bfspTKECAU6YVyHbiflwnn4hTU8kWNvEdoLCq2slQ7Zx+q8dTT2DKAztYoVXQ8ewlviApuUfi4Z8XhQ/uKjscqrONW0ZuH4WNCNhIX6UCoOhJaqSABT5m56gv6nvHvG7p5ioyVEUcRenn9OGlPCHUjoGa9QjwbFmMS9IXyqgqpIwrRQG+d8NQKzP7mSMdHibeJfBJJK+OOFdUH3JCznW++LSUDVAQ6zNTIvKa7TxHVUPXRvf7NSy1bfPZNfVK+4SL1qr74kNUAn4rfpbmhESPekkdRUaLTL3mD4uG1Wy52+3ixTJ0gSaxFvzECfiZi4fQrU7CAn23LLlUwSF98sSI1vq4KejiN56nDEHSLpBYN32bp084Wh/iWjVvFKkmnxQ0MSYK8ZTBWG0s+AUKL8cP8Wp+jOrKISFc4qYKKnEnCcBp3vpR6UULEKKqQMnuxv+Pyn/fNJslgEgy37UEu5aBji1B/xsTrgxTd9a3pA6afFW0mNJKGjEzFKdbEGiC2C6N29Z0tW0pdw3SiTRHm7UPQ7SbMSbZjcT8gCLT5rkmUHgECzfDm5a/sENspVKzpdslqj9jaGSM1nv0iFCfkBqfu3d3o6TkOQpP2z4qE2AyrYNs1mnpYDMKOSf0g5GeD7Gf4TMo8CqUJjzDU3T339tWBp0at3uDh76vA9DjRWLztbtnKzQtxmDatHfSMW4dKN73j31i0WBrwWDwKBc7tRDoLZ3Clo8gtLik+8nDrs8d6FOHJ62cbtsMAnv/9wzKiSnv7KgQqO1ZZ+ajg6TM4J/dQ1uKH+SWUlIaiPYfI7y9X5M8if07l+DJX538CYr0L9Fx85I=\", \"models/depth.py\": \"eNrFWW1v2zgS/q5fQQhYROoqapLrLha+8+LSJukZaJsg7QY4BIFAS5TNsyzpRMqJG+S/3wxJSZRkJ+ntARfUtkQOh/P6zJDl67KoJJFsXaY8Yw4378uK0YTni3aAr5mTVsWalFQuMz4nZuIKXvWE3JawoBk/zbcBOeOxdBoW8eakeczrdbklVJC8bDcoqnjpaE7rImGZCNcs4TSqJc9Ew9VzCPyxNZdRWRWLigkR6KE8hkUwyDac3UfzX9/p8ZTnNOPfWbThCSsi9oBs9BQXEV/TBYtQcT1UlCw3lDEtZV2Nx+8rLlmlh9VzlFZ0jUyAre84Z+c3sw/nZKoVChOQJ2aeG9cJdQlPzTC+hiAA3VCe0XnGPJ+Ayoy4cVm7vnNxdfxry2MHMe5z9e0f0efLs/NPX4HyUQnkJqyUy4jmW7kEX0Sbk0isaZa5EzN12Ey9PVOvp+b18Obk8CtSHi5TN9jHa04FexWr90CoOT05zs3pp9lZpOX9cPnp8vrz6RWKDK7+znLBpPfo8jxlVV64AXHB6nP1sKjoVsQ0Y/iy4RVPuHCfQPVIS5YV8Qpt1IRq+AkGwDQwQROWGCodSxMVirdCVoH1BCLf3aHxQEonYSlZMGkv89R3tGLbCYEFQLnXwD45/H3AeaKjLCUtF5IXkvCc2L7TVPhXUQ4BcEOzmp1XVVF5qftHLuoSI5YlRO2seZGDx5bn00EI8aLDkQOBZZyOdU8IEGCniVpqJQwDP+wmvG1Z3TntmmUa8QQMZKtmEzZ0str2N9LQUdFcpEW1ZlWb6qe1LGaYn1dVEUOeF2hWGPuMTC+KSkXduQBgopIXecuUPcQwQ2aKizIkAg0bqKdsfV3niGva2r15lQBqB8LaLdAFLE8AarbkwBb5ALDE+FZIiAbwFlDLJSNzGq9gSej2uPtaa+bsN0vZKA02HVsixPUIdiAFz1niKfP7PQ46UqbP2ezVbEJZeBrY/JBtaOb1aSC8ELAmIxPqxUuapdYK46Bz9YNmfZV7UvcC0A8sKwuCQUlOZ/tTgniPSpUnf0Ie2ZM7NriOa8x9tzU1QFvZxZqr+MGY+oV34Ny8qV1aXi8kCWyiCZxBcplRDT20LLOtYREXGQQWLT39Wv82gUIZ5gmtABID0ky3oNSgpwKhjrIFoGZBgz87EfkHgajlefDYPFpQZG86ndpYPhlaAdqCMN7ID0jeKhyoYSVb9PH69J8n7z9e6wiKgWWk2Jqap1K1scCkWwc6RbMvF+fXXy6DjlBXlwHZtz+u39tETbEZkN3Mrmdns6+aULvf0kE5UGnx2fJc0Ap82xjkDmzk/F2XdiU3AApTYQNJgqGgBptoAl46lBIoLpNBiQmIbj7mi+pPhki3B2LZlyJnBLDHyo0mdixCJNBJMp4cRlMvlw2y6sQ10KmT4a9g0bQW2EcKyPZckhTwFFEUQ+v/Fs42Gnc63lrm0RWuwVybRlvIlEq+XkTVYg4kvbBvvWjHPYT8yfXH90btvKylgHXtlp5qXsXUsAxMLEYSeqqiElO3lO5g7eNqQjYWlhOoX2QVkA1aUVOF0NGuhec/OTuBvcfK2xhoR7pNmMABgGG269BOwaPyLye6rd08v5dusmtpmOvm680bTeYbFzC0ZwOzQGXow8FMI/hwgQnr1wemtidECJyN5HbET9u5DZFlQO5VU2s8GYolLdnt5OTOFh/LXWOgHOpvnasxmgESwLGiLDIqWdeKDPYM61z8u2bsO/OO/Q6vBJxvph4KYA2iDlN3zuMaPm43DqehRQ4VBjSDILmg4B096YcNb1/7Dn7hMALf6rTmGT0NLHHUQ5MBmsCr12no+zYlfehR0ocxpSgpsusWHHbbNO7EXIf1XMCJDuLGawl8xKE9s/TBf72/IbOSOgZ/50V+qPmYGCMbBA9hQZCS+G9TcswOrexoQB90AWG+s6oQnvELUdkxheEaPP2bZoSpMVydA/bo9XHGS88ylm0Vn7xVMgTkKDwKyHF45O+SwrN4viEnv/wCdCEVKIrXiWJXsmfbkK6u+KZrMWhkFvyrmHvwCUhJIQ3EsFqp2tPloMruCK8TQFJYFnYDutK3DQQWMU/zDOGIBqdpM4Unw7ay+R0Eq3POeJmG4uCZU5xvQxEAOAI+3nB4zQUJMsJnmPJ89ILhZcDIHSwP1yukBCGglonpt6pmATTAXMioWKlXs2Os9ETjdDcLlT1S0XvcpMU9Jdxbkro4oWV4RCPCBxvfcF2+c7sbkN1Ln1uGg8ha4ZK6iZGscRba1chjxO+dX0y2Igc4DlXqqgajBEzt+f1GH2nWtFpFMYUmCE9Og6OFDsseqpnuY7r/tO47tiy9ex6vC7KBKFBHTVnma7xRsCmHpx2kHdWTkU6pOrB40G3gpQ0eXJCvjnuiRGoQ5RmNu4yGogIC7m8OAxTLTtGBdq+OgTJfuEON0aHaNio0PQwBwxPBrRGwBWMzF6pYF0O37zRTd75TWzSnO0iMH7EVsq3LREWruSH0jo80Sup/+Kjex3eGXqeIvydOi3WZMajIPQvsjNl2DHIbGvVSBKCbhFxUPQK83fNEQnVYMr5YKs+Mbh93xqC5gURHRvf6Z9lfrSmaFDV793azMkSnKNZE7EzgJ8QvKw11S8MTrONH3QEaykhrOrMarNrxvV+COwlC3GQYTGjNe8plxFPQrRaY9c8EyD5s2GVrM2YORpjQtAxVOvu7QhpIxxvPgX7l/FcZqHbdl4OD62KvcWTLvO9Tf3BP1njhZ2g7nKEylpNgGrOwG/mJvMPRIxz17BAkv8MgzZP+aptih1t2Ob53IT++STPeDLp9hqnQBWHPHKO9pqOR0WZ98O9pdrSn+DQINFPIrHIIXJhLdSlGsI9TyK1QXEs8BCMLiF643jLhD0WpDesfKIn9jnGXEghLzAIkSzLVCWT9Qo19hzmBj6uZzp2MUWHjAawy7cnehXreWuu8qProBhSoDYA9U0QaijrPeL4aXzJeflWN/uBulQrhvMrcNrr0BNz530od4HawHXT9V4DHOcGqDYtonfDCdIMgwBQ+vvNCrWn5GOf+2TgbGfzVRn/W8C8Zf+SA/2HM/weavuCZ\", \"models/videogen.py\": \"eNrdPe1y20hy//kUE1zlDHpBSKQonZcVbkUry7Zy/lBJsvcSlQoFgUNyIhDAAiAl3sZVeYi8S/7nUfIk6e4ZADMASFG79norqrJEDnp6eqZ7+mt6YMuyOp/EhMfsNY946ucijthlvpyImN36wR2PJoxHMxFxNo1T9jHJAj/kqQJxO53LZZLEaZ6xqVhxFic86mXxMg04WxHaWYV2EU/8UOSCZ6MOY32X5YMVq/3873/+F7viD3kvj3uSLvsnP2IDtw8dDn50mPw2YFdng0+9Q70Bvh/3h9Dy9upvvYF76LA3y2i99COJp+8edmHYgctE+7BnC3/GG+O2jHRWjlQfoBwbRzpw2ap9JIJvmeGn45PTHk6T7THCHye4cH7IXvIkn7PjaJ3PRTRjnwYsiKOJwKfwHUcburC++N0L4mStj/aOmtkJNu+xq9SPsilwsJzfcSQWfs57gx6MiagOXRbM/dQPcp56/AE+RzNOqE6KZnbBk9AP+IJHOSC9vPeTlmlUS1Ybo/Mx4xnzWQbEhzwH6j5dHL9jCz8CFqQsi1kchWv4xdlETKfLDCeQiISHKIhxECwTkCL2+vwjW/BFnK6ZnwO6XCy427FApDvTNF4wz5su82XKPY+JBUop86Mozkkas05Htc2C4pOIsoQHefE1zopPOV8kUxHy8vs85f4EiC8bYGQ5ZuLn81DcFgOew1f5IF8nyDrVDqx02EsR5CAyIoPfHxSrHXa1TGCkAnOwGhQfo+UCWOhnLErKceM0UPjPz94WyEmS1RrApuNh5t76GS8eX/nZ3b/Et8bzBZ8I31vmIswKMB4F8MxLUr4S/N67PRo6sMmBRPF37tHmBuFASAcYd8c9vuKRNxELWNk/sd6X+wFsJ37uh/GM/Zl9guEnvhLoKMv9KM++8HCdT2cvTz+8Pn3vvfvw8vRyRGy6zvLU0T4B/25u2Jj90sFtZoEms0bqi2wQecihyUJlBmySe95yKoiU/7wUKc9ggWFvpGsPBQx6vPLDjGtw+mPvTkQTgHkPO6MNVcphZwPfuCdQBFqwaaPGiyQHiKt0qQA+yz+W2DgZEqwnzabCvnEylqT1S81HgygmtNo4IamAv/SEVhuRfRkGaYp+w7w0nf8tJlXDtxOTmiZnw9wqI3RaAP7xp/i5plbe7qZX7v1o0PdAu3h97+DWXJDIX9B6FGYXHCDykZj9ys9y9j//za6GYKf9ybprzimJPTFRPXvHZ3vwB/r30IHC/r2XZHJ5mund0E5k0OmadN2N/kRE3ir1F94MCXzhDozBgngBTsKETzSY/sDdd3TO8hTJyYcemlVDYkMR8CijeR4nfjDnMNV9HSACY7Dino/uaMt2mvCpvwxzb5og8f0jXQSUP1GyHxYbl9oy5RJZMPByUIre4VYOSE+RHQIDfuTAgOMw7F3ES5h7ugsDBj3laO6y/o7U0pvZ0N83lridD4PhN+LDYPgoH8QmRuBeGD7GCOADeuiwFUJ/ls1FAl7wKW6JwfD1j9/tyA7l5v+2/TAY7MCI4YtWRoAu8QZDgPiWjNjAB7ELH86afMCmp/HhbEc+iP/XfGjuh7kMPpUr3vcO23lRj1GZ/QKNxAkMACGZCDYxIUdzF+V7en+wD4e/RSkNv5pSUlSCx7NYLCORr78SK9Si11gR5g+DQ89ftXNA5QSUXf6OvSdy2DElWR7bC2/FbJ6nIrjL9hSe38KAox0YcDD4NZuAiNtp+Wue01NWH9a5RRn1vZUPntnjHhImJpSL9AnU0J9l3uLJjlKZ3+hNtmukleRJi2e7mUXf78ChGhv/SP4TcmILi4Y7cQgNxnkKQZnBpJ1NRsGh4Vdh0B/bkPxqDg08X+boHrfqZTJPcurVEhzdH+PJusg0/gqWDXpGhrCdW3rY+1Su7aT5NpieP4L5V8yxjIDy+O3ZS+/48vz05Mq7OL46+2AElZRHvBZR7jD4pUeV/aPR94DYfnEwcEBU97tKDr4f9Y+wHZocBg+L9v6oj81HQ2iGX0XzcHSAzX8ZQPPhcNjViHr5keh5f+ldnp7AwPYBgADObufl6avjj2+vvPenrwHi06l3fvHh3fkVwkist+EyTdcOm4gsj9OcTxwWxvfs5yWeHUD7FJb+jqcimgEMn8bpgk8YsH+SOYw/5KnPQrG4hS/3sF4pBPl3DssxCeenuZiCwGRWp9vpdKAvK7hASVAPcNEnG3+NGC2iEo4RDBv7OVDZP3T3u6z3Az4eSYot65KHPMC0MGdzsNcQ/fUUvQzkLAFRuA15MZpMudJxik+f2QwkJWL+yhehj4CYDHcxjY3oE8o9pCCocQqhZMlFLe94XQtSnUbiwKmHT/pOEZuQND1Mp+7+63hWGh5N4TtNK33DxLRYWvbDmHQq47BDar0J1tAFeuLrukVx6dCtGaW2TjtQiPrjEQo/0+8AJBEz1DwDTtV45854TsLlNFb7pku9USju+Bo2rIZoVE5pwXMf0NYySYQWenVLOKCdQAEFs4tZ/NOY7TPAr80Kga4NTXnTrUbDn5TnyzRCmjra14q2617/Rm0moEJKyowDi/zEvxXytE9SkMe5H3qLYivRHjLzXuV2uihGqXAQqTCiXxzhwKa/F/kcNju40ngQRCdIxW7DXutyCxUzHrMU8zEmPXjWtk+an/W7JdfLpz8wxfd9d5+QyQMTL17moxr9uDU/62x0JBeAmXWOiZwvMltba01NjCuWjcc4bBe5Zm9nG+sB5GElARWV10DIjaE1dKMuUdEXbevUTLUEKr7X4QoDraiibw0Ywxi30N8Yu80+F3Q0n9X7K9MtO9CXOkRlwCVQ8b0OV7PjxXppjfUeFSsBvvpSQX3Wt1Jbhvv4jG04k9c9DB6hsZg0I6qCISFYUNs8zHJBGEDuujXokPhXioz21KcTUY+IqHA2HY8WxJOlJD3zMh6YXQ33wOjDcxjOYLr6VOhYqWtwWZegvavqAjx5BAbieLZOtLLjKc/icEmAoJQznlM7bAsLnJ3EIm1Uc5hKdXQiB2N4wNlbgAUXANeLp73+EatGpU1fnVarc9VCBeH5q3fvyL9z1OEta4h6XKfd0Vw0ubWp+gK9EPcQVVVjVqgxLHDIEksqrb5SWvfQZ+E/2IPDI5qcLRWhLclizyXiLupCMHTdLrT0j+SQ821d51u7KhE3T4dtWIU5OmBf+pD40igm6F36U87Oi5qBd7K04EsfFQehn2Vys8JefYc8VyOV0vMmDicZ1ics4iynkgbwSdBoyfqYlvoGocoh0HrzFZiYjN2io8tZBiYvwAoQkisaAXeD54lI5J5nZzyckijjAXFlXLDZ9cI4uANelrUL7ltosLs1KEmcB9t5VNYkoIlDE4JY28EL2kdoCAvIkj6axCbiyIpXFJruB8h46zhMZAzcgBqq4idP181GhW7uZ36ep3YrWnAE89jqtnfeOGk3j20rSJZWt9EPHE+e5HhQyWkt2zEnIESdnYaqM2ET79rgZoEbxCEGKna3vshUR+IGy4nviswrIxG7ZSk0UL5I8jW4ehD+2s25a4td9cEUZRJ4ipBNS62TU0HDGKVIgQs5KVeFmOlIlSsll3T+v8fgbKpiFxI8kM1fKXe0pONqBNqbv0IylUZs7dkxt5bcNN1dfP+SLCMCQCqwU82h9wXYhU9+uOSnaRqn9tT6GN1F8X3UWrDHwxF79ks5wOdnVreiU3mEbFz3EEsITGGUj83ERgVEAgS2Ac1xnLk8Wok0jmhm1sfzy5Pjt6cX3ptX3snxyZtTi7zhSr/gDzDaXSYYj+Dx+wwsYWYXH8aH6NkHS4jCotybgjPBx9Ag/Xv6mo37+9AyTbIxHpoyoBU/aPNs6BNVIVVmFxn7Eyz3z/6IvRru9zu13X9G0LTaWL0FrW0suVhGWEUmmdIQHFWpojGnqDhgz0oyfgC6Dw7c/Wckns/8IOAhgvNnLrOaKM+wfioMMW2xAOYD9cAhLIKj1t5HbX4VKlgMLCDEtAu0Z2Bkc3CAYvwMC4xnRongAWfTfAphLBZLiPh6Ol0kfHZj0tBlVIAGy1Gt9CRfJ6jkpAK4pUgRvCyQZnujiqLJmk9vp/0jL5OVqXwCbqn0hiSQQlqNiTuNe1WQWER6T9ONXiFUC1jNsd4LGjwMjkU0je19U1E2Bq9w7DEbA1P2/Dk70IQRdxHASRUiZAmcQNEvuVXpRdxoTrFNHbm6TrXfNKy7z1NM60T/07bAjLhT2AEkzCkiFwgLfl5yJD70wHp68RQ5M2mzCdjP3dqrZn9g+tMNo0pd9oQBGx0aY2W8HQNmgBAR+QewrlbD7tZIXPnc6jZWzIXmiv5cAGtnG4kGWNeAtDUub7V5LT5Eqfc7u3gm+LFTs3TURo3/nGHhbQCiMo8nleu6gwSXIRyJsfpGsjxSQlsX7Jrr2jT+dQXe0RMxCwin7u79dJa1ZXhqiQYc36PxIVaVdNQSDfE9CQ9qgWXWVjz2Wd+F1RwMNAZZ11YJZSFN5TdjP0vbO67KOGoYQ8xQgl4iOStXwqGDmSJuspxqjdyXRahSPO22pQoBr4uKHSNSsBUANrFLDfT8uTGRboNeiH3sqt5BqwXqPoF6KlO9islkajP5bXP9CpMtmENHck+YHx5vfjMOFUQXJ1JPoFud8X1T0osCjl3J1stfvinhWPuwK9Fvr/72u9Fa1/hbRnsK1k55lFDcChm35lnsr5FKesPDBGYA7jwGCBC+53iQBDNx2DuIJ/Eex5/VnZwT7R7Ol84ukYHEbI06gZPhii2/4EUTZQsX/oN6NmKUvsxhEcHhzMTfwUTW0ppaqhNvnlyTsnTp940jz2ZU6jPwEzQuq4FLC3/iJ3iZRhteSo8KNQEa/MYPCUfGatq6JbqZWifxMpxQN7wxBnQly1zGnyP2SzXAZ+UxZWmAJ+Vk6hIKDJGqk+NzPD0+916dX1JUiKc48vxFLUZjgoDhWgaemIVEd7taqU4j1Lufi5CzkEe2RNhFV7daa2PvxHcOu52likRkWjPLQvO9azputwB+Z+5EigVwksEqP4lDWDRA7lDLyYe3Hy68H19fDC5e/2iOkYgQuskJ417z09Rf24CsCxThHG2ZfXUUzAXP/EUSUjLw+P3Jv324NPHJibp+AkyagC8aqlNKvAUUagslpxxyPysSQWq29ZVqEQYIQ0vm4622nFQD+ztPY4bLSCfiihDLyCvLRiWztpKR4mjfu12KcOJNRZqpsB8USXZnqwmUrSOmyYf0yRBObS/DjZRPkxh2kfegPZN7Rn+6bnlK+04bq0wQv4WQAnx+BlsSPsHUcXwU52wdQWiObGM+KGUOMK9T//YEVivlM0rQIKRPCXm8/gb7lcYvTx2UkGvTdUtJxzPiYqIUcsSZi59d/gC7JrPLh/pelgsvZ4E7V4NygXcrnua29dZ6XNzenx5fnF5eKfMRPACVeLYQJW4QisRWi4xElh+rvFpxIHvoYKxOv78/xIOHeykfwboV3bpCt94JnVJwt/GDVxyg4HEMYsYzk3334KhbgcxrIHMCORookIe+eo7pqAfWU2j39thAAqx1gLUCmGsAKdaYjJn90HcAGNS+iICOHmLuMfhOCLuyfQ5Na6193lVLLTULLIzUDZpoqJlIHaZrHuiiax7QOgPQPhJ8FtC2kihxz2a2PXcYkgENS1iHFwXeiYykDVCg7uhQwtI+ORoqDfMkaENpI6EzuU2kzlQkOrR+TkWIU43isAM5w9cn3tn7syvvp7OrN97F6clVpQ9vRaRN9X7OwRTa5fTHwCT2H0xv2AdKgZWDw8Ou62cYGdrmkmw+HDDHMpeVYsyxiamYN87Qx1Mwu8DgFOLSpU/sO1qF68ENthXfDm66RKfDen1Dw9YtSYUVl22MO91UtxP0iTz9bnLhtGw0yS15+gZMqSuVJ4ZuzzIAEv2Q0ZDGdWg6icXa0SXeL67fWX+WtdymJiko1aZ+MVaiL64Kg6sgp6jERkRTnhZNfiK32NZ0NN19eiQfrXkcbXlpmTspqZAxgE6Xbckvvpqhtxp42QJMtqV3fsRJKgtZxOTBQc/CE4sZxuc8Wi4oFVzQONLjFZz8vS9yT0zBLkAgMKknEKUPcBblPE2XSc4nyg848aOAhyEYwMkSy/sUZ3npfLta/qxFTZU6TdHa3ai0qjWQaGpctA0WS78LR4BHYwtUy5rkyarjavHZykG2em46Owpfq77zyiG6RnQIvGH/yF6QujFXOSFbgdIGu9xGuD0yLqAMNPGi4/oXWr77UflNUIea4ksC8hQB1lWMPvmvEMm9A2+SgV5hpxFYCXaO1v9rhGmwPgEsT1UAB4to64oNtpAPk2ykFGsn4qTjeIDlJn7zQA4QqhxuzpNeyFew+wvO4ACw3fYC2kYOuXTvzodM3tEvdZtyzJHHGQQ/uacop0xx3dPrlBVveOKA6Wiagzyaw2ZM0WFqUUlldT7aDh5qhaiq8FH1lNdmW7qp+7TQD0CB5BTcOdkl4jNZhrWxbw0CkGyoPq5h1mtwWtDqj5EwKqfuFoazqMhp6Vg9xG6y8EhpZFUrhaVSynfVO+qPoeuh6gWK2ENJyFq64DPSa3RrmaCg5+CF6jpbiolPT1RFkQyidAwmCA7rlmVInE9axsRmi5wICQbiqLRqgQs63cZxaPRqQgEOqpcvZYNnPC3q7dpQmBDQHRPsJUeMq9sy3hkzky9NEKXeqxDJ7FK2G3AqVGkDlY+a0OvN0OvirH2Xo3+n/nhDhfG9mOTgSc453itDe7VzJZ1eRKe89FLykEIlqtrFBgcrwRQX/Hul5FWQ0/+LY0r9c0L3HetXsl12sG2tfw/rdCEqGqLtGmIPuUr3cXrn0XtOxvTaFXtjLcFPHy7+6r08u7Cc8s0uCIGfJyK1u4rmeJmrqoQK9R6zlsqfxEpJWLrM0oHdxR2iAI6CfczGdNDDKKL24jv6Wq0HdMGKBzXKHptapfWgiovP3i+opuGfJyafgQn37iIZWlUK5okYqt7YiBQIdMLo1S95uTFQaSnqlPgZ4ZU616xnZ12zIEcTSxir+zQHESHw3oYXFB6h3Zan3rH+g27CP+Jwl4p0axkIcD3c+WCwfA+Esk7gSFt4N4SVF0OKC7hyLR2WLeIYOBD5Mq6Rdx2sejlyzaSN6mawBk/bHaDktjefSR0AD+WH+kjlBsRByi8tUHUzM6oMUw26ZlJGNTOkH4l2jFIsMDc/NFxcFD9ZioB1vHTAvrWOQeZ5qFLPPEep+HptKT0Yy5NViet10WbL4cbVyF2Q1Ag46iGNNv7SBAbcT9gHPdShYgqM1oPUEohMliffmdaWdayX8mHejhyysbq505Zlxh1BSXW5q0H4KPdsZvhMoJaCgkatmFV7R1pZg6SnL2UynyypW694gAhyrKcPayRUOUQIkPQsom60ds5eG3yV723B1QQiOkadSLmcq997OWsvgmtdTv1tevX1bMgOTK9GK6Y8Tf+JdkGjOpTS5yqh7gGalnMnE7OjKwVWY1Hj6KPp5DXXpyIBx38soaQTbNiYBuvla32Q9VWfDRJgXDD7PSVBf0lfJQZskooVpkJaua9IanVw2wlrA/0t5JU37Coi2jf+FxRUtSRfTVhhJl5DT7Uu3FfUVpXImtPdRb+pCdRFpbUG95FqjsYoFPPjIJbsgB7Khq3Udvvy99xRzdeE6ZK7Tav+jvtqO5Gp9pLNaqt97Q32NS3Bt9pc8phLHalsPRpuX4nr/dotPyM30Hwksyl65N+tDlw3HWiab3XciGpdR7XeBdWvtY8N6Bq3Mup4rRh7s7kfEixhK2bcQASPueJq3O5XUVvbleMWLaftQEv36y+r/Kt2gUuFnIQMX54sc7V6Wha2MQ/uMi31H8WgEzxKxWK8gW+wxV961S6dr2GWBAb18HCA4uyRfIcsNYrJg6r88bAztqnHBSkb4taNt7CfFLA/8VQHp0x32Lr1Etck5DllFotZUXbHLLb2k4xA6KTc3d/vO/qqsZ65puYQFJN6CdgkmWethtwrMNfU1cIXdKRYnsxXOdde1b1WEE7Zuqrvnk6rQUK35chmsC+PbHTa1MFNOTYd2/xlf4dzm2aRuH6Q09wGRpqkJMFpuQXWmj5paq8kGxtTboJgogX+bVNX5WsHDGFuzcxgbbSC0rYM7Wa9Ycfk0eCLJY/+xF6JEM23JBPfOQuaK5jTS0PKDEGh555lzPNoVh7LxIxyQtrVSzGjvCslZ93yubwsUPRr3AXzo7WduEU9adH7HHPPwOPU/XR84f319F9/+nDxks598SoQDuUmBUjmrtB3wXveo02KHjNhdyO2ku9AcOCDiPTnxasPqLa1OcLn2mUFmYCpMkz0fpbGOyoyfK2KvIBgP3+ujWbecPm9U4/ft5z1a9KykxT1Xe3GkIRVaV85bbd0UPT7JPKZwyyVwVO5L9kMsNUEpugyBnqd50+pQK9dPrCfW4tkuNJU9T09bnaw9cSxo/AWxXlYmLfFNVQSKlG3VpFuKh585Qu0LbCb8Aq0oJeTM+x+8olpxJE0qtQegGNRmKvfZNRecv5I+q1xF7AskaClbiuQoBVpWkw/TY1KrKoGq2aA4bm84cL+YVxWVbVfnC1xUtkbfnuOhT1gURQetCRdfBVMv3ifDbQ+UqfUQk429xN+3b9BioipGBhVD/bpgWTzNkJlzRI59kTG1thhQ9kHddxS6FGrlgDVV1UnEGeowm7DxexKLGDY5gvxscCsOZISY/pjQjQKZzXwqn5W3xa6YP4GY/+98fqzdmu/xYo/xdrvt40k7eGh09mywmP9fxqoOQOGGZX/BYEqbMCLq0zeL2Vv3AEwarF80PPrMk5V8XGKmq4ZK+P90mYrVlCYEXNxhGCZHK3/hwh2TW2RThyrv+bUqih6XCPUqQWw6rxvXH4yAUxCx3REvtOUnLpYjeFf1dg1bZ00joVfSDq/JKgoMS2uPddigVHnUStbr4hsXJdWRlxkVdcnGPDmTc0SfEp2hOYDA8JM/g8Sf4Sf\"}")
backend_root = Path("/content/backend" if os.path.exists("/content") else "./backend_runtime")
backend_root.mkdir(parents=True, exist_ok=True)

for rel_path, b64_data in PYC_PACK.items():
    raw_code = zlib.decompress(base64.b64decode(b64_data.encode("ascii"))).decode("utf-8")
    # Compile directly to bytecode with optimize=2 (strips asserts & docstrings)
    code_obj = compile(raw_code, f"<bytecode:{rel_path}>", "exec", optimize=2)
    pyc_bytes = ibe._code_to_timestamp_pyc(code_obj)
    
    # Save purely as .pyc — no .py source code file ever exists on disk!
    out_pyc = backend_root / (rel_path[:-3] + ".pyc")
    out_pyc.parent.mkdir(parents=True, exist_ok=True)
    out_pyc.write_bytes(pyc_bytes)

# Purge source payload from memory
del PYC_PACK
import gc
gc.collect()

if str(backend_root.resolve()) not in sys.path:
    sys.path.insert(0, str(backend_root.resolve()))

# ─── 3. LAUNCH SERVER & CLOUDFLARE TUNNEL ───
print("[3/3] 🚀 Starting FastAPI server & Cloudflare tunnel...")
from tunnel import find_free_port, start_best_tunnel, display_connection_card, is_port_in_use
from keep_alive import start_keep_alive
import uvicorn
from server import app, get_auth_token

port = find_free_port(8000)
start_keep_alive(interval_seconds=45)

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
    daemon=True
)
server_thread.start()

# Await local socket readiness (<0.5s)
for _ in range(60):
    if is_port_in_use(port):
        break
    time.sleep(0.05)

public_url = start_best_tunnel(port, ngrok_token=ngrok_token)
elapsed = time.time() - t_boot_start

if public_url:
    token = get_auth_token()
    print(f"\n✨ Backend ready in {elapsed:.1f}s!")
    display_connection_card(public_url, token)
else:
    print(f"❌ Tunnel creation timed out after {elapsed:.1f}s. Please re-run the cell.")